In [0]:
"""
Imports PySpark SQL functions for use in subsequent transformations. These functions are used for column operations, conditional logic, and string manipulation.
"""
from pyspark.sql.functions import col, when, regexp_extract, regexp_replace, translate, lit, coalesce

from pyspark.sql import DataFrame

from typing import List

In [0]:
%sql
---SQL cell: Shows all tables in the 'googleads_bronze' database to verify available raw data sources.
SHOW TABLES IN googleads_bronze;

database,tableName,isTemporary
googleads_bronze,ad_copy_and_landing_page_performance,false
googleads_bronze,ad_group_ad_asset_view,false
googleads_bronze,ads_data,false
googleads_bronze,ads_performance,false
googleads_bronze,asset,false
googleads_bronze,audience_age_performance,false
googleads_bronze,audience_gender_performance,false
googleads_bronze,call_lead_generation,false
googleads_bronze,campaign_asset,false
googleads_bronze,competitive_impression_share,false


In [0]:
"""
Reads all Delta tables from the bronze layer into Spark DataFrames and displays the first 3 rows of each for inspection.
Each DataFrame represents a different aspect of Google Ads reporting (campaign, keyword, search term, etc.).
"""
# Read Delta tables
core_campaign_df = spark.table("googleads_bronze.core_campaign_performance")  # Core campaign metrics
# display(core_campaign_df.limit(3))  # Display sample rows

search_keyword_df = spark.table("googleads_bronze.search_keyword_performance")  # Keyword-level metrics
# display(search_keyword_df.limit(3))

search_term_df = spark.table("googleads_bronze.search_term_analysis")  # Search term metrics
# display(search_term_df.limit(3))

conversion_df = spark.table("googleads_bronze.conversion_performance")  # Conversion metrics
# display(conversion_df.limit(3))

ad_copy_df = spark.table("googleads_bronze.ad_copy_and_landing_page_performance")  # Ad copy and landing page metrics
# display(ad_copy_df.limit(3))

optimization_device_df = spark.table("googleads_bronze.optimization_device_and_time")  # Device and time metrics
# display(optimization_device_df.limit(3))

audience_gender_df = spark.table("googleads_bronze.audience_gender_performance")  # Gender audience metrics
# display(audience_gender_df.limit(3))

audience_age_df = spark.table("googleads_bronze.audience_age_performance")  # Age audience metrics
# display(audience_age_df.limit(3))

competitive_impression_df = spark.table("googleads_bronze.competitive_impression_share")  # Impression share metrics
# display(competitive_impression_df.limit(3))

network_df = spark.table("googleads_bronze.network_performance")  # Network-level metrics
# display(network_df.limit(3))

display_ad_df = spark.table("googleads_bronze.display_ad_viewability")  # Display ad viewability metrics
# display(display_ad_df.limit(3))

call_lead_generation_df = spark.table("googleads_bronze.call_lead_generation")  # Call lead generation metrics
# display(call_lead_generation_df.limit(3))

geographic_df = spark.table("googleads_bronze.geographic_performance")  # Geographic performance metrics
# display(geographic_df.limit(3))

ads_performance_df = spark.table("googleads_bronze.ads_performance")  # Ad performance metrics
# display(ads_performance_df.limit(3))

ads_data_df = spark.table("googleads_bronze.ads_data")  # Ad data details
# display(ads_data_df.limit(3))

campaign_asset_df = spark.table("googleads_bronze.campaign_asset")  # Campaign asset metrics
# display(campaign_asset_df.limit(3))

ad_group_ad_asset_view_df = spark.table("googleads_bronze.ad_group_ad_asset_view")  # Ad group asset view metrics
# display(ad_group_ad_asset_view_df.limit(3))

asset_df = spark.table("googleads_bronze.asset")  # Asset data
# display(asset_df.limit(3))

conversion_action_df = spark.table("googleads_bronze.conversion_action")  # Conversion action details
# display(conversion_action_df.limit(3))

placement_performance_df = spark.table("googleads_bronze.placement_performance")  # Placement performance metrics
# display(placement_performance_df.limit(3))

group_placement_performance_df = spark.table("googleads_bronze.group_placement_performance")  # Group placement performance
# display(group_placement_performance_df.limit(3))

geo_target_constant_df = spark.table("googleads_bronze.geo_target_constant")  # Geo target constant data
# display(group_placement_performance_df.limit(3))

In [0]:
"""
Displays the full DataFrame and schema for the core campaign performance data for further inspection.
Useful for understanding the structure and sample data before transformation.
"""
display(core_campaign_df.limit(3))
core_campaign_df.printSchema()

campaign.id,segments.date,campaign.name,campaign.status,campaign.advertising_channel_type,metrics.impressions,metrics.clicks,metrics.ctr,metrics.cost_micros,metrics.average_cpc
21771245709,2024-11-22,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,112056,1179,0.010521524951809809,3.92312256E8,332750.00508905854
22973632076,2025-09-08,Data Engineering (AWS-Sept),PAUSED,PERFORMANCE_MAX,34546,1422,0.04116250796040063,1.381432353E9,971471.4156118144
21771245709,2025-12-22,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,22702,3060,0.13478988635362524,9.93128612E8,324551.8339869281


root
 |-- campaign.id: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- campaign.status: string (nullable = true)
 |-- campaign.advertising_channel_type: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.ctr: double (nullable = true)
 |-- metrics.cost_micros: double (nullable = true)
 |-- metrics.average_cpc: double (nullable = true)



In [0]:
# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in core_campaign_df.columns]  # Clean column names for Spark compatibility

# Define lists for handling nulls using the NEW names.
# numeric_cols_to_fill: List of numeric columns to fill with 0 if null
# string_cols_to_fill: List of string columns to fill with 'Unknown' if null
numeric_cols_to_fill = [
    'metrics_impressions', 'metrics_clicks', 'metrics_ctr',
    'metrics_cost_micros', 'metrics_average_cpc'
]
string_cols_to_fill = [
    'campaign_id', 'campaign_name', 'campaign_status',
    'campaign_advertising_channel_type'
]

# --- Step 2: Chain all transformations into a final DataFrame ---
"""
Renames columns, handles nulls, calculates cost in INR, and selects final columns for the core campaign performance DataFrame.
This block prepares the campaign metrics for analytics and reporting.
"""
df_core_campaign = core_campaign_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("ctr_pct",
        when(col("metrics_impressions") > 0, (col("metrics_clicks") / col("metrics_impressions")) * 100)
        .otherwise(0)
    ) \
    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000  # Convert micro currency to INR
    ) \
    .withColumn("cpc_inr",
        when(col("metrics_clicks") > 0, col("cost_inr") / col("metrics_clicks"))
        .otherwise(0)
    ) \
    .select(
        "campaign_id",
        "segments_date",
        "campaign_name",
        "campaign_status",
        "campaign_advertising_channel_type",
        "metrics_impressions",
        "metrics_clicks",
        "ctr_pct",
        "cpc_inr",
        "cost_inr"  # Keep the new cost column
        # "metrics_cost_micros" is now omitted
    )


# --- Step 3: Display the final, cleaned output ---
"""
Displays the schema and sample data for the transformed core campaign DataFrame.
This is a verification step to ensure the transformation is correct.
"""
print("--- Final Schema ---")
df_core_campaign.printSchema()

print("\n--- Final Transformed Data ---")
display(df_core_campaign.limit(10))

--- Final Schema ---
root
 |-- campaign_id: string (nullable = false)
 |-- segments_date: date (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- campaign_status: string (nullable = false)
 |-- campaign_advertising_channel_type: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cost_inr: double (nullable = true)


--- Final Transformed Data ---


campaign_id,segments_date,campaign_name,campaign_status,campaign_advertising_channel_type,metrics_impressions,metrics_clicks,ctr_pct,cpc_inr,cost_inr
21771245709,2024-11-22,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,112056,1179,1.0521524951809809,0.3327500050890585,392.312256
22973632076,2025-09-08,Data Engineering (AWS-Sept),PAUSED,PERFORMANCE_MAX,34546,1422,4.116250796040063,0.9714714156118143,1381.432353
21771245709,2025-12-22,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,22702,3060,13.478988635362525,0.3245518339869281,993.128612
21771245709,2025-02-06,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,21020,2364,11.246431969552807,0.1662898130287648,393.109118
21771245709,2024-11-26,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,20317,1092,5.374809273022592,0.3924442060439561,428.549073
21771245709,2025-02-05,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,19923,1448,7.267981729659188,0.4327837962707182,626.670937
21771245709,2025-02-11,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,19718,2763,14.012577340501064,0.1400872797683677,387.061154
22489991406,2025-07-09,Hyderabad,PAUSED,SEARCH,18200,466,2.5604395604395607,1.7333274506437768,807.730592
21771245709,2025-02-10,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,17397,2237,12.858538828533655,0.172835767098793,386.633611
21771245709,2025-12-20,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,16908,1440,8.516678495386799,0.34262517430555556,493.380251


In [0]:
"""
Displays the first 3 rows and schema of the search keyword performance DataFrame for inspection.
"""
display(search_keyword_df.limit(3))
search_keyword_df.printSchema()

campaign.id,ad_group.id,ad_group_criterion.criterion_id,segments.date,campaign.name,ad_group.name,ad_group_criterion.keyword.text,ad_group_criterion.keyword.match_type,ad_group_criterion.quality_info.quality_score,metrics.impressions,metrics.clicks,metrics.cost_micros
21781297084,169807400833,13507497083,2024-11-30,Azure Data Engineering -Display - text,Ad group 1,online computer courses with certificate,BROAD,null,12,0,0.0
21781297084,169807400833,297748590701,2024-11-30,Azure Data Engineering -Display - text,Ad group 1,data engineering courses,BROAD,null,55,0,0.0
21781297084,169807400833,342470401435,2024-11-30,Azure Data Engineering -Display - text,Ad group 1,naresh it hyderabad,BROAD,null,13,0,0.0


root
 |-- campaign.id: string (nullable = true)
 |-- ad_group.id: string (nullable = true)
 |-- ad_group_criterion.criterion_id: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- ad_group.name: string (nullable = true)
 |-- ad_group_criterion.keyword.text: string (nullable = true)
 |-- ad_group_criterion.keyword.match_type: string (nullable = true)
 |-- ad_group_criterion.quality_info.quality_score: integer (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.cost_micros: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning ---
"""
Renames columns, handles nulls, calculates metrics (CTR, CPC, cost_inr), and selects final columns for the search keyword performance DataFrame.
"""
new_column_names = [c.replace('.', '_').replace('__', '_') for c in search_keyword_df.columns]

numeric_cols_to_fill = [
    'ad_group_criterion_quality_info_quality_score',
    'metrics_impressions',
    'metrics_clicks',
    'metrics_cost_micros'
]
string_cols_to_fill = [
    'campaign_name',
    'ad_group_name',
    'ad_group_criterion_keyword_text',
    'ad_group_criterion_keyword_match_type'
]

# --- Step 2: Chain all transformations for the final DataFrame ---
df_search_keyword = search_keyword_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("cost_inr", col("metrics_cost_micros") / 1000000) \
    .withColumn("ctr_pct",
        when(col("metrics_impressions") > 0, (col("metrics_clicks") / col("metrics_impressions")) * 100)
        .otherwise(0)
    ) \
    .withColumn("cpc_inr",
        when(col("metrics_clicks") > 0, col("cost_inr") / col("metrics_clicks"))
        .otherwise(0)
    ) \
    .select(
        "segments_date",
        "campaign_id",   
        "ad_group_id",   
        "campaign_name",
        "ad_group_name",
        col("ad_group_criterion_keyword_text").alias("keyword"),
        col("ad_group_criterion_keyword_match_type").alias("keyword_match_type"),
        col("ad_group_criterion_quality_info_quality_score").alias("quality_score"),
        "metrics_impressions",
        "metrics_clicks",
        "ctr_pct",
        "cpc_inr",
        "cost_inr"
    )

# --- Step 3: Display the final, cleaned output ---
"""
Displays the schema and sample data for the transformed search keyword DataFrame.
"""
print("--- Final Keyword Schema ---")
df_search_keyword.printSchema()

print("\n--- Final Transformed Keyword Data ---")
display(df_search_keyword.limit(10))

--- Final Keyword Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- ad_group_id: string (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- ad_group_name: string (nullable = false)
 |-- keyword: string (nullable = false)
 |-- keyword_match_type: string (nullable = false)
 |-- quality_score: integer (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- ctr_pct: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- cost_inr: double (nullable = true)


--- Final Transformed Keyword Data ---


segments_date,campaign_id,ad_group_id,campaign_name,ad_group_name,keyword,keyword_match_type,quality_score,metrics_impressions,metrics_clicks,ctr_pct,cpc_inr,cost_inr
2025-07-12,22489991406,177495674774,Hyderabad,Ad group 1,Data Analytics,BROAD,0,2474,272,10.994341147938561,6.1803103749999995,1681.044422
2025-07-11,22489991406,177495674774,Hyderabad,Ad group 1,Data Analytics,BROAD,0,1688,188,11.137440758293838,6.031182457446809,1133.862302
2025-05-07,22489991406,177495674774,Hyderabad,Ad group 1,it training,BROAD,0,1304,78,5.98159509202454,4.175793192307692,325.711869
2025-04-30,22489991406,177495674774,Hyderabad,Ad group 1,it training,BROAD,0,410,68,16.585365853658537,3.6008688529411765,244.859082
2025-05-09,22489991406,177495674774,Hyderabad,Ad group 1,it training,BROAD,0,1158,63,5.4404145077720205,2.8125765396825395,177.192322
2025-05-07,22489991406,177495674774,Hyderabad,Ad group 1,Data Analytics,BROAD,0,1133,59,5.207413945278023,6.109341813559322,360.451167
2025-05-08,22489991406,177495674774,Hyderabad,Ad group 1,Data Analytics,BROAD,0,1029,59,5.733722060252672,4.302245847457627,253.832505
2025-05-01,22489991406,177495674774,Hyderabad,Ad group 1,Data Analytics,BROAD,0,594,58,9.764309764309765,4.131320534482759,239.616591
2025-05-08,22489991406,177495674774,Hyderabad,Ad group 1,it training,BROAD,0,2094,57,2.722063037249284,3.608047087719298,205.658684
2025-05-02,22489991406,177495674774,Hyderabad,Ad group 1,Data Analytics,BROAD,0,943,55,5.832449628844114,5.430000109090909,298.650006


In [0]:
"""
Displays the first 3 rows and schema of the search term analysis DataFrame for inspection.
"""
display(search_term_df.limit(3))
search_term_df.printSchema()

campaign.id,ad_group.id,segments.date,campaign.name,ad_group.name,search_term_view.search_term,search_term_view.status,segments.search_term_match_type,segments.search_term_match_source,segments.keyword.ad_group_criterion,segments.keyword.info.text,segments.keyword.info.match_type,metrics.impressions,metrics.clicks,metrics.cost_micros,metrics.conversions,metrics.average_cpc,metrics.ctr
22489991406,177495674774,2025-07-13,Hyderabad,Ad group 1,online courses for computer science engineering students in india,NONE,NEAR_PHRASE,ADVERTISER_PROVIDED_KEYWORD,customers/1401815809/adGroupCriteria/177495674774~12133691,Education,BROAD,1,1,1780000.0,0.0,1780000.0,1.0
22489991406,177495674774,2025-07-13,Hyderabad,Ad group 1,online programs,NONE,NEAR_PHRASE,ADVERTISER_PROVIDED_KEYWORD,customers/1401815809/adGroupCriteria/177495674774~12133691,Education,BROAD,2,1,1.857E7,0.0,1.857E7,0.5
22489991406,177495674774,2025-04-29,Hyderabad,Ad group 1,analytics dashboard,NONE,NEAR_PHRASE,ADVERTISER_PROVIDED_KEYWORD,customers/1401815809/adGroupCriteria/177495674774~12411881,data analytics,BROAD,1,1,1900000.0,0.0,1900000.0,1.0


root
 |-- campaign.id: string (nullable = true)
 |-- ad_group.id: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- ad_group.name: string (nullable = true)
 |-- search_term_view.search_term: string (nullable = true)
 |-- search_term_view.status: string (nullable = true)
 |-- segments.search_term_match_type: string (nullable = true)
 |-- segments.search_term_match_source: string (nullable = true)
 |-- segments.keyword.ad_group_criterion: string (nullable = true)
 |-- segments.keyword.info.text: string (nullable = true)
 |-- segments.keyword.info.match_type: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.cost_micros: double (nullable = true)
 |-- metrics.conversions: double (nullable = true)
 |-- metrics.average_cpc: double (nullable = true)
 |-- metrics.ctr: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning ---
"""
Renames columns, handles nulls, extracts criterion_id, calculates metrics, and selects final columns for the search term analysis DataFrame.
"""

# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in search_term_df.columns]

# Define lists for handling nulls using the NEW, cleaned names
numeric_cols_to_fill = [
    'metrics_impressions',
    'metrics_clicks',
    'metrics_conversions',
    'metrics_cost_micros',
    'metrics_average_cpc',
    'metrics_ctr'
]

string_cols_to_fill = [
    'campaign_name',
    'ad_group_name',
    'search_term_view_search_term',
    'search_term_view_status',
    'segments_search_term_match_type',
    'segments_keyword_ad_group_criterion',
    'segments_keyword_info_text',
    'segments_keyword_info_match_type'
]

# --- Step 2: Chain all transformations for the final DataFrame ---
# This renames, cleans nulls, extracts criterion_id, calculates new metrics, and selects final columns.

df_search_term = search_term_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn(
        "triggering_keyword_criterion_id",
        # Extract criterion_id from resource name
        # Format: "customers/XXX/adGroupCriteria/YYY~ZZZ"
        # Extract the "ZZZ" part (criterion_id after the tilde)
        regexp_extract(
            col("segments_keyword_ad_group_criterion"), r'adGroupCriteria/\d+~(\d+)', 1)
    ) \
    .withColumn("cost_inr", col("metrics_cost_micros") / 1000000) \
    .withColumn("cpc_inr", col("metrics_average_cpc") / 1000000) \
    .withColumn("ctr_pct",
        when(col("metrics_impressions") > 0, (col("metrics_clicks") / col("metrics_impressions")) * 100)
        .otherwise(0)
    ) \
    .select(
        "segments_date",
        "campaign_id",
        "ad_group_id",
        "campaign_name",
        "ad_group_name",
        
        # Search term details
        col("search_term_view_search_term").alias("search_term"),
        col("search_term_view_status").alias("search_term_status"),
        col("segments_search_term_match_type").alias("search_term_match_type"),
        
        # Triggering keyword details
        "triggering_keyword_criterion_id",
        col("segments_keyword_info_text").alias("triggering_keyword_text"),
        col("segments_keyword_info_match_type").alias("triggering_keyword_match_type"),
        
        # Raw metrics
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        
        # Calculated metrics (cost and engagement-focused)
        "cost_inr",
        "cpc_inr",
        "ctr_pct"
        
        # Note: Removed CVR and CPL calculations since conversion data has zeros/nulls
        # Can add back later when conversion tracking is fixed
    )

# --- Step 3: Display the final, cleaned output ---

print("--- Final Search Term Schema (with Triggering Keyword Attribution) ---")
df_search_term.printSchema()

# print("\n--- Sample Data Showing Search Term → Keyword Mapping ---")
# print("Ordered by cost:")
# display(df_search_term.select(
#     "search_term",
#     "triggering_keyword_text",
#     "triggering_keyword_match_type",
#     "search_term_match_type",
#     "metrics_clicks",
#     "cost_inr",
#     "ctr_pct"
# ).orderBy(col("cost_inr").desc()))

# print("\n--- Verification: Check Keyword Attribution ---")
# print("Unique triggering keywords:")
# display(df_search_term.select("triggering_keyword_text", "triggering_keyword_match_type") \
#     .distinct() \
#     .orderBy("triggering_keyword_text"))

print("\n--- Full Transformed Search Term Data ---")
display(df_search_term.limit(10))

--- Final Search Term Schema (with Triggering Keyword Attribution) ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- ad_group_id: string (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- ad_group_name: string (nullable = false)
 |-- search_term: string (nullable = false)
 |-- search_term_status: string (nullable = false)
 |-- search_term_match_type: string (nullable = false)
 |-- triggering_keyword_criterion_id: string (nullable = false)
 |-- triggering_keyword_text: string (nullable = false)
 |-- triggering_keyword_match_type: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)
 |-- cpc_inr: double (nullable = true)
 |-- ctr_pct: double (nullable = true)


--- Full Transformed Search Term Data ---


segments_date,campaign_id,ad_group_id,campaign_name,ad_group_name,search_term,search_term_status,search_term_match_type,triggering_keyword_criterion_id,triggering_keyword_text,triggering_keyword_match_type,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr,cpc_inr,ctr_pct
2025-05-01,22489991406,177495674774,Hyderabad,Ad group 1,ai programming courses,NONE,NEAR_PHRASE,334008193914,Software Training,BROAD,240,41,0.0,103.635015,2.5276832926829265,17.083333333333332
2025-05-08,22489991406,177495674774,Hyderabad,Ad group 1,ai programming courses,NONE,NEAR_PHRASE,334008193914,Software Training,BROAD,303,34,0.0,112.668464,3.3137783529411764,11.221122112211221
2025-05-08,22489991406,177495674774,Hyderabad,Ad group 1,데이터 분석 서비스,NONE,NEAR_PHRASE,129465465,Data Analytics,BROAD,178,33,0.0,56.025276,1.6977356363636364,18.53932584269663
2025-04-30,22489991406,177495674774,Hyderabad,Ad group 1,ai programming courses,NONE,NEAR_PHRASE,334008193914,Software Training,BROAD,184,32,0.0,88.159565,2.75498640625,17.391304347826086
2025-05-09,22489991406,177495674774,Hyderabad,Ad group 1,ai programming courses,NONE,NEAR_PHRASE,334008193914,Software Training,BROAD,210,32,0.0,94.873194,2.9647873125,15.238095238095239
2025-05-02,22489991406,177495674774,Hyderabad,Ad group 1,ai programming courses,NONE,NEAR_PHRASE,334008193914,Software Training,BROAD,204,17,0.0,59.063757,3.474338647058824,8.333333333333332
2025-04-30,22489991406,177495674774,Hyderabad,Ad group 1,technology and business success,NONE,BROAD,11101161,it training,BROAD,64,16,0.0,50.566069,3.1603793125,25.0
2025-05-25,22489991406,177495674774,Hyderabad,Ad group 1,professional cloud security engineer training,NONE,NEAR_PHRASE,11101161,it training,BROAD,112,14,0.0,32.137531,2.2955379285714286,12.5
2025-05-04,22489991406,177495674774,Hyderabad,Ad group 1,ai programming courses,NONE,NEAR_PHRASE,334008193914,Software Training,BROAD,100,13,0.0,37.925706,2.917362,13.0
2025-02-27,22271705916,175636335299,Leads-Search-24Feb2025,Ad group 1,data analytics courses in hyderabad with placements,NONE,BROAD,848336226199,learn python for data engineering,BROAD,71,12,0.0,154.42,12.868333333333334,16.901408450704224


In [0]:
"""
Displays the first 3 rows and schema of the conversion performance DataFrame for inspection.
"""
display(conversion_df.limit(3))
conversion_df.printSchema()

campaign.id,segments.date,campaign.name,campaign.status,segments.device,segments.conversion_action,segments.conversion_action_name,segments.conversion_action_category,metrics.conversions,metrics.conversions_value,metrics.all_conversions,metrics.all_conversions_value,metrics.value_per_conversion,metrics.value_per_all_conversions
23393627674,2026-02-03,Azure-Data-Engineering-Jan-2026,ENABLED,MOBILE,customers/1401815809/conversionActions/7122187177,Local actions - Other engagements,ENGAGEMENT,0.0,0.0,30.0,30.0,null,1.0
22998241842,2025-09-22,10-Sept AWS Snowflake,PAUSED,MOBILE,customers/1401815809/conversionActions/7122187177,Local actions - Other engagements,ENGAGEMENT,0.0,0.0,17.0,17.0,null,1.0
22489991406,2025-06-18,Hyderabad,PAUSED,MOBILE,customers/1401815809/conversionActions/7122187177,Local actions - Other engagements,ENGAGEMENT,0.0,0.0,14.0,14.0,null,1.0


root
 |-- campaign.id: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- campaign.status: string (nullable = true)
 |-- segments.device: string (nullable = true)
 |-- segments.conversion_action: string (nullable = true)
 |-- segments.conversion_action_name: string (nullable = true)
 |-- segments.conversion_action_category: string (nullable = true)
 |-- metrics.conversions: double (nullable = true)
 |-- metrics.conversions_value: double (nullable = true)
 |-- metrics.all_conversions: double (nullable = true)
 |-- metrics.all_conversions_value: double (nullable = true)
 |-- metrics.value_per_conversion: double (nullable = true)
 |-- metrics.value_per_all_conversions: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning (Corrected) ---

# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in conversion_df.columns]

# Define lists for handling nulls using the REAL, cleaned names
numeric_cols_to_fill = [
    'metrics_all_conversions',
    'metrics_all_conversions_value',
    'metrics_conversions',
    'metrics_conversions_value',
    'metrics_value_per_conversion',
    'metrics_value_per_all_conversions'
]
# Corrected: Use the actual column name from your data
string_cols_to_fill = [
    'segments_conversion_action',
    'segments_conversion_action_name', 'segments_conversion_action_category'
]


# --- Step 2: Chain all transformations for the final DataFrame (Corrected) ---
# This renames, cleans nulls, calculates new metrics, and selects final columns.
df_conversion = conversion_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("value_per_all_conversions",
        when(col("metrics_all_conversions") > 0, col("metrics_all_conversions_value") / col("metrics_all_conversions"))
        .otherwise(0)
    ) \
    .withColumn("value_per_conversion",
        when(col("metrics_conversions") > 0, col("metrics_conversions_value") / col("metrics_conversions"))
        .otherwise(0)
    ) \
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        # Corrected: Select the actual column and give it a clean alias
        col("segments_conversion_action").alias("conversion_action"),
        col("segments_conversion_action_name").alias("conversion_action_name"),
        col("segments_conversion_action_category").alias("conversion_action_category"),
        col("metrics_conversions").alias("conversions"),
        col("metrics_conversions_value").alias("conversions_value"),
        "value_per_conversion",
        col("metrics_all_conversions").alias("all_conversions"),
        col("metrics_all_conversions_value").alias("all_conversions_value"),
        "value_per_all_conversions"
    )

# --- Step 3: Display the final, cleaned output ---

print("--- Final Conversion Schema ---")
df_conversion.printSchema()

print("\n--- Final Transformed Conversion Data ---")
display(df_conversion.limit(10))

--- Final Conversion Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- conversion_action: string (nullable = false)
 |-- conversion_action_name: string (nullable = false)
 |-- conversion_action_category: string (nullable = false)
 |-- conversions: double (nullable = false)
 |-- conversions_value: double (nullable = false)
 |-- value_per_conversion: double (nullable = true)
 |-- all_conversions: double (nullable = false)
 |-- all_conversions_value: double (nullable = false)
 |-- value_per_all_conversions: double (nullable = true)


--- Final Transformed Conversion Data ---


segments_date,campaign_id,campaign_name,conversion_action,conversion_action_name,conversion_action_category,conversions,conversions_value,value_per_conversion,all_conversions,all_conversions_value,value_per_all_conversions
2026-02-03,23393627674,Azure-Data-Engineering-Jan-2026,customers/1401815809/conversionActions/7122187177,Local actions - Other engagements,ENGAGEMENT,0.0,0.0,0.0,30.0,30.0,1.0
2025-09-22,22998241842,10-Sept AWS Snowflake,customers/1401815809/conversionActions/7122187177,Local actions - Other engagements,ENGAGEMENT,0.0,0.0,0.0,17.0,17.0,1.0
2025-06-18,22489991406,Hyderabad,customers/1401815809/conversionActions/7122187177,Local actions - Other engagements,ENGAGEMENT,0.0,0.0,0.0,14.0,14.0,1.0
2025-07-14,22489991406,Hyderabad,customers/1401815809/conversionActions/7122187177,Local actions - Other engagements,ENGAGEMENT,0.0,0.0,0.0,12.0,12.0,1.0
2025-05-14,22489991406,Hyderabad,customers/1401815809/conversionActions/7122187177,Local actions - Other engagements,ENGAGEMENT,0.0,0.0,0.0,9.0,9.0,1.0
2026-01-09,23393627674,Azure-Data-Engineering-Jan-2026,customers/1401815809/conversionActions/7315415268,Lead form - Submit,SUBMIT_LEAD_FORM,9.0,9.0,1.0,9.0,9.0,1.0
2026-01-13,23393627674,Azure-Data-Engineering-Jan-2026,customers/1401815809/conversionActions/7315415268,Lead form - Submit,SUBMIT_LEAD_FORM,9.0,9.0,1.0,9.0,9.0,1.0
2026-01-18,23393627674,Azure-Data-Engineering-Jan-2026,customers/1401815809/conversionActions/7315415268,Lead form - Submit,SUBMIT_LEAD_FORM,9.0,9.0,1.0,9.0,9.0,1.0
2025-09-17,22998241842,10-Sept AWS Snowflake,customers/1401815809/conversionActions/7122187177,Local actions - Other engagements,ENGAGEMENT,0.0,0.0,0.0,8.0,8.0,1.0
2026-01-10,23393627674,Azure-Data-Engineering-Jan-2026,customers/1401815809/conversionActions/7122187177,Local actions - Other engagements,ENGAGEMENT,0.0,0.0,0.0,8.0,8.0,1.0


In [0]:
"""
Displays the first 3 rows and schema of the ad copy and landing page performance DataFrame for inspection.
"""
display(ad_copy_df.limit(3))
ad_copy_df.printSchema()

campaign.id,ad_group.id,ad_group_ad.ad.id,segments.date,campaign.name,ad_group.name,ad_group_ad.ad.final_urls,metrics.impressions,metrics.clicks,metrics.conversions
21771245709,169993022084,715897121402,2025-12-22,Azure Data Engineering -Display - Image,Ad group 1,https://academyofdata.ai,22702,3060,0.0
21771245709,169993022084,715897121402,2025-02-11,Azure Data Engineering -Display - Image,Ad group 1,https://academyofdata.ai,19718,2763,0.0
21771245709,169993022084,715897121402,2025-02-06,Azure Data Engineering -Display - Image,Ad group 1,https://academyofdata.ai,21020,2364,0.0


root
 |-- campaign.id: string (nullable = true)
 |-- ad_group.id: string (nullable = true)
 |-- ad_group_ad.ad.id: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- ad_group.name: string (nullable = true)
 |-- ad_group_ad.ad.final_urls: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.conversions: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning ---

# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in ad_copy_df.columns]

# Define lists for handling nulls using the NEW, cleaned names
numeric_cols_to_fill = [
    'metrics_impressions',
    'metrics_clicks',
    'metrics_conversions'
]
string_cols_to_fill = [
    'campaign_name',
    'ad_group_name',
    'ad_group_ad_ad_final_urls'
]


# --- Step 2: Chain all transformations for the final DataFrame (Alternative Method) ---
# This renames, cleans nulls, calculates new metrics, and selects final columns.
df_ad_copy = ad_copy_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("ctr_pct",
        when(col("metrics_impressions") > 0, (col("metrics_clicks") / col("metrics_impressions")) * 100)
        .otherwise(0)
    ) \
    .withColumn("cvr_pct",
        when(col("metrics_clicks") > 0, (col("metrics_conversions") / col("metrics_clicks")) * 100)
        .otherwise(0)
    ) \
    .withColumn(
        "domain",
        regexp_extract(col("ad_group_ad_ad_final_urls"), r'https?://([^/]+)', 1)
    ) \
    .select(
        "segments_date",
        "campaign_id",
        "ad_group_id",
        "campaign_name",
        "ad_group_name",
        col("ad_group_ad_ad_id").alias("ad_id"),
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        "ctr_pct",
        "cvr_pct",
        col("ad_group_ad_ad_final_urls").alias("final_url"),
        "domain"
    )

# --- Step 3: Display the final, cleaned output ---

print("--- Final Ad Copy Schema ---")
df_ad_copy.printSchema()

print("\n--- Final Transformed Ad Copy Data ---")
display(df_ad_copy.limit(10))

--- Final Ad Copy Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- ad_group_id: string (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- ad_group_name: string (nullable = false)
 |-- ad_id: string (nullable = true)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- metrics_conversions: double (nullable = false)
 |-- ctr_pct: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- final_url: string (nullable = false)
 |-- domain: string (nullable = false)


--- Final Transformed Ad Copy Data ---


segments_date,campaign_id,ad_group_id,campaign_name,ad_group_name,ad_id,metrics_impressions,metrics_clicks,metrics_conversions,ctr_pct,cvr_pct,final_url,domain
2025-12-22,21771245709,169993022084,Azure Data Engineering -Display - Image,Ad group 1,715897121402,22702,3060,0.0,13.478988635362525,0.0,https://academyofdata.ai,academyofdata.ai
2025-02-11,21771245709,169993022084,Azure Data Engineering -Display - Image,Ad group 1,715897121402,19718,2763,0.0,14.012577340501064,0.0,https://academyofdata.ai,academyofdata.ai
2025-02-06,21771245709,169993022084,Azure Data Engineering -Display - Image,Ad group 1,715897121402,21020,2364,0.0,11.246431969552807,0.0,https://academyofdata.ai,academyofdata.ai
2025-02-10,21771245709,169993022084,Azure Data Engineering -Display - Image,Ad group 1,715897121402,17397,2237,0.0,12.858538828533655,0.0,https://academyofdata.ai,academyofdata.ai
2025-02-09,21771245709,169993022084,Azure Data Engineering -Display - Image,Ad group 1,715897121402,16058,2063,0.0,12.847178976211234,0.0,https://academyofdata.ai,academyofdata.ai
2025-02-07,21771245709,169993022084,Azure Data Engineering -Display - Image,Ad group 1,715897121402,14983,1928,0.0,12.86791697256891,0.0,https://academyofdata.ai,academyofdata.ai
2025-02-08,21771245709,169993022084,Azure Data Engineering -Display - Image,Ad group 1,715897121402,14476,1805,0.0,12.468914064658746,0.0,https://academyofdata.ai,academyofdata.ai
2024-11-29,21771245709,169993022084,Azure Data Engineering -Display - Image,Ad group 1,715897121402,15686,1804,0.0,11.50070126227209,0.0,https://academyofdata.ai,academyofdata.ai
2024-11-23,21771245709,169993022084,Azure Data Engineering -Display - Image,Ad group 1,715897121402,14310,1772,0.0,12.382948986722573,0.0,https://academyofdata.ai,academyofdata.ai
2024-11-30,21771245709,169993022084,Azure Data Engineering -Display - Image,Ad group 1,715897121402,13607,1639,0.0,12.045270816491511,0.0,https://academyofdata.ai,academyofdata.ai


In [0]:
"""
Displays the first 3 rows and schema of the optimization device and time DataFrame for inspection.
"""
display(optimization_device_df.limit(3))
optimization_device_df.printSchema()

campaign.id,segments.date,segments.device,segments.day_of_week,segments.hour,metrics.impressions,metrics.clicks,metrics.conversions,metrics.cost_micros,metrics.ctr,metrics.average_cpc
21519858184,2024-09-03,TABLET,TUESDAY,19,25,1,0.0,250384.0,0.04,250384.0
21519858184,2024-09-03,TABLET,TUESDAY,20,39,1,0.0,4012183.0,0.02564102564102564,4012183.0
21519858184,2024-09-10,DESKTOP,TUESDAY,10,5,1,0.0,630000.0,0.2,630000.0


root
 |-- campaign.id: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- segments.device: string (nullable = true)
 |-- segments.day_of_week: string (nullable = true)
 |-- segments.hour: integer (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.conversions: double (nullable = true)
 |-- metrics.cost_micros: double (nullable = true)
 |-- metrics.ctr: double (nullable = true)
 |-- metrics.average_cpc: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning ---

# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in optimization_device_df.columns]

# Define lists for handling nulls using the NEW, cleaned names
numeric_cols_to_fill = [
    'segments_hour',
    'metrics_impressions',
    'metrics_clicks',
    'metrics_conversions',
    'metrics_cost_micros',
    'metrics_ctr',
    'metrics_average_cpc'
]
string_cols_to_fill = [
    'segments_device',
    'segments_day_of_week'
]

# --- Step 2: Chain all transformations for the final DataFrame ---

df_optimization_device = optimization_device_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("ctr_pct",
        when(col("metrics_impressions") > 0, (col("metrics_clicks") / col("metrics_impressions")) * 100)
        .otherwise(0)
    ) \
    .withColumn("cvr_pct",
        when(col("metrics_clicks") > 0, (col("metrics_conversions") / col("metrics_clicks")) * 100)
        .otherwise(0)
    ) \
    .withColumn("cost_inr", col("metrics_cost_micros") / 1000000) \
    .withColumn("cpc_inr", col("metrics_average_cpc") / 1000000) \
    .withColumn("time_of_day",
        when((col("segments_hour") >= 6) & (col("segments_hour") <= 11), "Morning")
        .when((col("segments_hour") >= 12) & (col("segments_hour") <= 17), "Afternoon")
        .when((col("segments_hour") >= 18) & (col("segments_hour") <= 23), "Evening")
        .otherwise("Night")
    ) \
    .select(
        "segments_date",
        "campaign_id",
        col("segments_device").alias("device"),
        col("segments_day_of_week").alias("day_of_week"),
        col("segments_hour").alias("hour"),
        "time_of_day",
        "metrics_impressions",
        "metrics_clicks",
        "ctr_pct",
        "metrics_conversions",
        "cvr_pct",
        "cost_inr",
        "cpc_inr"
    )

# --- Step 3: Display the final, cleaned output ---

print("--- Final Optimization Schema ---")
df_optimization_device.printSchema()

print("\n--- Final Transformed Optimization Data ---")
display(df_optimization_device.limit(10))

--- Final Optimization Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- device: string (nullable = false)
 |-- day_of_week: string (nullable = false)
 |-- hour: integer (nullable = false)
 |-- time_of_day: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- ctr_pct: double (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cvr_pct: double (nullable = true)
 |-- cost_inr: double (nullable = true)
 |-- cpc_inr: double (nullable = true)


--- Final Transformed Optimization Data ---


segments_date,campaign_id,device,day_of_week,hour,time_of_day,metrics_impressions,metrics_clicks,ctr_pct,metrics_conversions,cvr_pct,cost_inr,cpc_inr
2025-12-22,21771245709,MOBILE,MONDAY,8,Morning,19276,2602,13.498651172442417,0.0,0.0,823.355771,0.31643188739431205
2024-11-30,21771245709,MOBILE,SATURDAY,14,Afternoon,11687,1430,12.235817575083425,0.0,0.0,332.660388,0.23262964195804195
2025-02-08,21771245709,MOBILE,SATURDAY,16,Afternoon,10850,1337,12.32258064516129,0.0,0.0,334.414614,0.25012312191473446
2025-02-05,21771245709,MOBILE,WEDNESDAY,15,Afternoon,16472,1270,7.710053423992229,0.0,0.0,400.266802,0.31517071023622045
2025-09-08,22973632076,MOBILE,MONDAY,11,Morning,27368,1250,4.567377959660917,0.0,0.0,887.808796,0.7102470368
2025-02-06,21771245709,MOBILE,THURSDAY,9,Morning,11120,1250,11.241007194244604,0.0,0.0,209.9077,0.16792616
2025-02-07,21771245709,MOBILE,FRIDAY,12,Afternoon,9180,1198,13.050108932461873,0.0,0.0,235.206612,0.19633273121869782
2024-11-28,21771245709,MOBILE,THURSDAY,6,Morning,14770,1132,7.664184157075152,0.0,0.0,338.338795,0.29888586130742045
2024-11-21,21771245709,MOBILE,THURSDAY,14,Afternoon,11202,1070,9.551865738261025,0.0,0.0,361.617353,0.33796014299065424
2024-11-06,21771245709,MOBILE,WEDNESDAY,6,Morning,7759,891,11.483438587446836,0.0,0.0,502.966241,0.564496342312009


In [0]:
"""
Displays the first 3 rows and schema of the audience gender performance DataFrame for inspection.
"""
display(audience_gender_df.limit(3))
audience_gender_df.printSchema()

campaign.id,ad_group.id,ad_group_criterion.criterion_id,segments.date,campaign.name,ad_group_criterion.gender.type,metrics.impressions,metrics.clicks,metrics.cost_micros,metrics.conversions
21771245709,169993022084,10,2025-02-11,Azure Data Engineering -Display - Image,MALE,12090,1764,2.45692111E8,0.0
21771245709,169993022084,10,2025-02-06,Azure Data Engineering -Display - Image,MALE,15564,1717,2.86321935E8,0.0
21771245709,169993022084,10,2025-12-22,Azure Data Engineering -Display - Image,MALE,11609,1610,5.54651903E8,0.0


root
 |-- campaign.id: string (nullable = true)
 |-- ad_group.id: string (nullable = true)
 |-- ad_group_criterion.criterion_id: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- ad_group_criterion.gender.type: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.cost_micros: double (nullable = true)
 |-- metrics.conversions: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning ---

# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in audience_gender_df.columns]

# Define lists for handling nulls using the NEW, cleaned names
numeric_cols_to_fill = [
    'metrics_impressions',
    'metrics_clicks',
    'metrics_conversions',
    'metrics_cost_micros'
]
string_cols_to_fill = [
    'campaign_name',
    'ad_group_criterion_gender_type'
]

# --- Step 2: Chain all transformations for the final DataFrame ---

df_audience_gender = audience_gender_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("ctr_pct",
        when(col("metrics_impressions") > 0, (col("metrics_clicks") / col("metrics_impressions")) * 100)
        .otherwise(0)
    ) \
    .withColumn("cvr_pct",
        when(col("metrics_clicks") > 0, (col("metrics_conversions") / col("metrics_clicks")) * 100)
        .otherwise(0)
    ) \
    .withColumn("cost_inr", col("metrics_cost_micros") / 1000000) \
    .select(
        "segments_date",
        "campaign_id",
        "ad_group_id",
        "campaign_name",
        col("ad_group_criterion_gender_type").alias("gender"),
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        "ctr_pct",
        "cvr_pct",
        "cost_inr"
    )

# --- Step 3: Display the final, cleaned output ---

print("--- Final Gender Schema ---")
df_audience_gender.printSchema()

print("\n--- Final Transformed Gender Data ---")
display(df_audience_gender.limit(10))

--- Final Gender Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- ad_group_id: string (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- gender: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- metrics_conversions: double (nullable = false)
 |-- ctr_pct: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cost_inr: double (nullable = true)


--- Final Transformed Gender Data ---


segments_date,campaign_id,ad_group_id,campaign_name,gender,metrics_impressions,metrics_clicks,metrics_conversions,ctr_pct,cvr_pct,cost_inr
2025-02-11,21771245709,169993022084,Azure Data Engineering -Display - Image,MALE,12090,1764,0.0,14.590570719602978,0.0,245.692111
2025-02-06,21771245709,169993022084,Azure Data Engineering -Display - Image,MALE,15564,1717,0.0,11.031868414289386,0.0,286.321935
2025-12-22,21771245709,169993022084,Azure Data Engineering -Display - Image,MALE,11609,1610,0.0,13.868550262727194,0.0,554.651903
2025-02-10,21771245709,169993022084,Azure Data Engineering -Display - Image,MALE,11376,1553,0.0,13.651547116736989,0.0,265.785196
2025-02-07,21771245709,169993022084,Azure Data Engineering -Display - Image,MALE,10210,1358,0.0,13.300685602350637,0.0,272.706932
2024-11-29,21771245709,169993022084,Azure Data Engineering -Display - Image,MALE,9848,1149,0.0,11.667343623070673,0.0,242.691872
2024-10-09,21771245709,169993022084,Azure Data Engineering -Display - Image,MALE,9316,1098,0.0,11.786174323744095,0.0,418.274284
2025-02-09,21771245709,169993022084,Azure Data Engineering -Display - Image,MALE,8433,1098,0.0,13.020277481323372,0.0,228.27266
2024-11-23,21771245709,169993022084,Azure Data Engineering -Display - Image,MALE,8786,1074,0.0,12.223992715684043,0.0,224.841886
2024-11-20,21771245709,169993022084,Azure Data Engineering -Display - Image,MALE,10733,1072,0.0,9.987887822603186,0.0,283.237741


In [0]:
"""
Displays the first 3 rows and schema of the audience age performance DataFrame for inspection.
"""
display(audience_age_df.limit(3))
audience_age_df.printSchema()

campaign.id,ad_group.id,ad_group_criterion.criterion_id,segments.date,campaign.name,ad_group_criterion.age_range.type,metrics.impressions,metrics.clicks,metrics.conversions,metrics.cost_micros
21771245709,169993022084,503999,2025-12-22,Azure Data Engineering -Display - Image,AGE_RANGE_UNDETERMINED,8207,1048,0.0,3.09991564E8
21771245709,169993022084,503001,2025-02-06,Azure Data Engineering -Display - Image,AGE_RANGE_18_24,8203,811,0.0,1.26855206E8
21771245709,169993022084,503002,2024-11-17,Azure Data Engineering -Display - Image,AGE_RANGE_25_34,1537,739,0.0,2.49655791E8


root
 |-- campaign.id: string (nullable = true)
 |-- ad_group.id: string (nullable = true)
 |-- ad_group_criterion.criterion_id: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- ad_group_criterion.age_range.type: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.conversions: double (nullable = true)
 |-- metrics.cost_micros: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning ---

# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in audience_age_df.columns]

# Define lists for handling nulls using the NEW, cleaned names
numeric_cols_to_fill = [
    'metrics_impressions',
    'metrics_clicks',
    'metrics_conversions',
    'metrics_cost_micros'
]
string_cols_to_fill = [
    'campaign_name',
    'ad_group_criterion_age_range_type'
]

# --- Step 2: Chain all transformations for the final DataFrame ---

df_audience_age = audience_age_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("ctr_pct",
        when(col("metrics_impressions") > 0, (col("metrics_clicks") / col("metrics_impressions")) * 100)
        .otherwise(0)
    ) \
    .withColumn("cvr_pct",
        when(col("metrics_clicks") > 0, (col("metrics_conversions") / col("metrics_clicks")) * 100)
        .otherwise(0)
    ) \
    .withColumn("age_range",
        translate(
            regexp_replace(col("ad_group_criterion_age_range_type"), "AGE_RANGE_", ""),
            "_", "-"
        )
    ) \
    .withColumn("cost_inr", col("metrics_cost_micros") / 1000000) \
    .select(
        "segments_date",
        "campaign_id",
        "ad_group_id",
        "campaign_name",
        "age_range",
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        "ctr_pct",
        "cvr_pct",
        "cost_inr"
    )


# --- Step 3: Display the final, cleaned output ---

print("--- Final Age Schema ---")
df_audience_age.printSchema()

print("\n--- Final Transformed Age Data ---")
display(df_audience_age.limit(10))

--- Final Age Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- ad_group_id: string (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- age_range: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- metrics_conversions: double (nullable = false)
 |-- ctr_pct: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- cost_inr: double (nullable = true)


--- Final Transformed Age Data ---


segments_date,campaign_id,ad_group_id,campaign_name,age_range,metrics_impressions,metrics_clicks,metrics_conversions,ctr_pct,cvr_pct,cost_inr
2025-12-22,21771245709,169993022084,Azure Data Engineering -Display - Image,UNDETERMINED,8207,1048,0.0,12.769586937979774,0.0,309.991564
2025-02-06,21771245709,169993022084,Azure Data Engineering -Display - Image,18-24,8203,811,0.0,9.88662684383762,0.0,126.855206
2024-11-17,21771245709,169993022084,Azure Data Engineering -Display - Image,25-34,1537,739,0.0,48.08067664281067,0.0,249.655791
2025-02-06,21771245709,169993022084,Azure Data Engineering -Display - Image,25-34,5894,714,0.0,12.114014251781473,0.0,120.244595
2025-02-07,21771245709,169993022084,Azure Data Engineering -Display - Image,25-34,5011,676,0.0,13.490321293155057,0.0,133.742941
2025-12-22,21771245709,169993022084,Azure Data Engineering -Display - Image,25-34,4514,676,0.0,14.97563136907399,0.0,232.795686
2025-02-10,21771245709,169993022084,Azure Data Engineering -Display - Image,25-34,4915,652,0.0,13.265513733468973,0.0,106.448327
2025-02-11,21771245709,169993022084,Azure Data Engineering -Display - Image,25-34,4813,648,0.0,13.463536255973404,0.0,80.697222
2025-02-09,21771245709,169993022084,Azure Data Engineering -Display - Image,UNDETERMINED,4790,622,0.0,12.985386221294362,0.0,108.588025
2025-02-11,21771245709,169993022084,Azure Data Engineering -Display - Image,18-24,4404,609,0.0,13.828337874659399,0.0,80.063286


In [0]:
"""
Displays the first 3 rows and schema of the competitive impression share DataFrame for inspection.
"""
display(competitive_impression_df.limit(3))
competitive_impression_df.printSchema()

campaign.id,segments.date,campaign.name,metrics.search_impression_share,metrics.search_top_impression_share,metrics.search_absolute_top_impression_share
23393627674,2025-12-23,Azure-Data-Engineering-Jan-2026,0.1373913043478261,0.0999,0.0999
21781297084,2024-11-21,Azure Data Engineering -Display - text,0.10058111380145278,0.0999,0.0999
21781297084,2024-10-04,Azure Data Engineering -Display - text,0.0999,0.0999,0.0999


root
 |-- campaign.id: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- metrics.search_impression_share: double (nullable = true)
 |-- metrics.search_top_impression_share: double (nullable = true)
 |-- metrics.search_absolute_top_impression_share: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning ---

# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in competitive_impression_df.columns]

# Define lists for handling nulls using the NEW, cleaned names
numeric_cols_to_fill = [
    'metrics_search_impression_share',
    'metrics_search_top_impression_share',
    'metrics_search_absolute_top_impression_share'
]
string_cols_to_fill = [
    'campaign_name'
]


# --- Step 2: Chain all transformations for the final DataFrame ---

df_competitive_impression = competitive_impression_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("search_impression_share_pct", col("metrics_search_impression_share") * 100) \
    .withColumn("search_top_is_pct", col("metrics_search_top_impression_share") * 100) \
    .withColumn("search_abs_top_is_pct", col("metrics_search_absolute_top_impression_share") * 100) \
    .withColumn("search_lost_is_pct", (1 - col("metrics_search_impression_share")) * 100) \
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        col("search_impression_share_pct").alias("impression_share_pct"),
        col("search_top_is_pct").alias("top_impression_share_pct"),
        col("search_abs_top_is_pct").alias("abs_top_impression_share_pct"),
        col("search_lost_is_pct").alias("lost_impression_share_pct")
    )


# --- Step 3: Display the final, cleaned output ---

print("--- Final Competitive Impression Schema ---")
df_competitive_impression.printSchema()

print("\n--- Final Transformed Competitive Impression Data ---")
display(df_competitive_impression.limit(10))

--- Final Competitive Impression Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- impression_share_pct: double (nullable = false)
 |-- top_impression_share_pct: double (nullable = false)
 |-- abs_top_impression_share_pct: double (nullable = false)
 |-- lost_impression_share_pct: double (nullable = false)


--- Final Transformed Competitive Impression Data ---


segments_date,campaign_id,campaign_name,impression_share_pct,top_impression_share_pct,abs_top_impression_share_pct,lost_impression_share_pct
2025-12-23,23393627674,Azure-Data-Engineering-Jan-2026,13.73913043478261,9.99,9.99,86.26086956521739
2024-11-21,21781297084,Azure Data Engineering -Display - text,10.058111380145277,9.99,9.99,89.94188861985472
2024-10-04,21781297084,Azure Data Engineering -Display - text,9.99,9.99,9.99,90.01
2024-10-05,21781297084,Azure Data Engineering -Display - text,9.99,9.99,9.99,90.01
2024-10-06,21781297084,Azure Data Engineering -Display - text,9.99,9.99,9.99,90.01
2024-10-07,21781297084,Azure Data Engineering -Display - text,9.99,9.99,9.99,90.01
2024-10-08,21781297084,Azure Data Engineering -Display - text,9.99,9.99,9.99,90.01
2024-10-09,21781297084,Azure Data Engineering -Display - text,9.99,9.99,9.99,90.01
2024-10-10,21781297084,Azure Data Engineering -Display - text,9.99,9.99,9.99,90.01
2024-10-11,21781297084,Azure Data Engineering -Display - text,9.99,9.99,9.99,90.01


In [0]:
"""
Displays the first 3 rows and schema of the network performance DataFrame for inspection.
"""
display(network_df.limit(3))
network_df.printSchema()

campaign.id,segments.date,campaign.name,segments.ad_network_type,metrics.impressions,metrics.clicks,metrics.conversions
21771245709,2025-12-22,Azure Data Engineering -Display - Image,CONTENT,22702,3060,0.0
21771245709,2025-02-11,Azure Data Engineering -Display - Image,CONTENT,19718,2763,0.0
21771245709,2025-02-06,Azure Data Engineering -Display - Image,CONTENT,21020,2364,0.0


root
 |-- campaign.id: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- segments.ad_network_type: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.conversions: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning ---

# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in network_df.columns]

# Define lists for handling nulls using the NEW, cleaned names
numeric_cols_to_fill = [
    'metrics_impressions',
    'metrics_clicks',
    'metrics_conversions'
]
string_cols_to_fill = [
    'campaign_name',
    'segments_ad_network_type'
]

# --- Step 2: Chain all transformations for the final DataFrame ---

df_network = network_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("ctr_pct",
        when(col("metrics_impressions") > 0, (col("metrics_clicks") / col("metrics_impressions")) * 100)
        .otherwise(0)
    ) \
    .withColumn("cvr_pct",
        when(col("metrics_clicks") > 0, (col("metrics_conversions") / col("metrics_clicks")) * 100)
        .otherwise(0)
    ) \
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        col("segments_ad_network_type").alias("ad_network"),
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        "ctr_pct",
        "cvr_pct"
    )

# --- Step 3: Display the final, cleaned output ---

print("--- Final Ad Network Schema ---")
df_network.printSchema()

print("\n--- Final Transformed Ad Network Data ---")
display(df_network.limit(10))

--- Final Ad Network Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- ad_network: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- metrics_conversions: double (nullable = false)
 |-- ctr_pct: double (nullable = true)
 |-- cvr_pct: double (nullable = true)


--- Final Transformed Ad Network Data ---


segments_date,campaign_id,campaign_name,ad_network,metrics_impressions,metrics_clicks,metrics_conversions,ctr_pct,cvr_pct
2025-12-22,21771245709,Azure Data Engineering -Display - Image,CONTENT,22702,3060,0.0,13.478988635362525,0.0
2025-02-11,21771245709,Azure Data Engineering -Display - Image,CONTENT,19718,2763,0.0,14.012577340501064,0.0
2025-02-06,21771245709,Azure Data Engineering -Display - Image,CONTENT,21020,2364,0.0,11.246431969552807,0.0
2025-02-10,21771245709,Azure Data Engineering -Display - Image,CONTENT,17397,2237,0.0,12.858538828533655,0.0
2025-02-09,21771245709,Azure Data Engineering -Display - Image,CONTENT,16058,2063,0.0,12.847178976211234,0.0
2025-02-07,21771245709,Azure Data Engineering -Display - Image,CONTENT,14983,1928,0.0,12.86791697256891,0.0
2025-02-08,21771245709,Azure Data Engineering -Display - Image,CONTENT,14476,1805,0.0,12.468914064658746,0.0
2024-11-29,21771245709,Azure Data Engineering -Display - Image,CONTENT,15686,1804,0.0,11.50070126227209,0.0
2024-11-23,21771245709,Azure Data Engineering -Display - Image,CONTENT,14310,1772,0.0,12.382948986722573,0.0
2025-09-05,22973632076,Data Engineering (AWS-Sept),YOUTUBE,14341,1649,0.0,11.498500801896661,0.0


In [0]:
"""
Displays the first 3 rows and schema of the display ad viewability DataFrame for inspection.
"""
display(display_ad_df.limit(3))
display_ad_df.printSchema()

segments.date,campaign.name,ad_group.name,metrics.active_view_impressions,metrics.active_view_measurability,metrics.active_view_viewability,metrics.active_view_cpm
2024-10-04,Azure Data Engineering -Display - Image,Ad group 1,4691,0.9887477897444141,0.7626402211022598,6.530464421232147E7
2024-10-05,Azure Data Engineering -Display - Image,Ad group 1,2849,0.9697300245432233,0.8011811023622047,7.663080414180414E7
2024-10-06,Azure Data Engineering -Display - Image,Ad group 1,4482,0.976249760582264,0.8793407886992348,8.906343016510487E7


root
 |-- segments.date: date (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- ad_group.name: string (nullable = true)
 |-- metrics.active_view_impressions: integer (nullable = true)
 |-- metrics.active_view_measurability: double (nullable = true)
 |-- metrics.active_view_viewability: double (nullable = true)
 |-- metrics.active_view_cpm: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning ---

# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in display_ad_df.columns]

# Define lists for handling nulls using the NEW, cleaned names
numeric_cols_to_fill = [
    'metrics_active_view_impressions',
    'metrics_active_view_measurability',
    'metrics_active_view_viewability',
    'metrics_active_view_cpm'
]
string_cols_to_fill = [
    'campaign_name',
    'ad_group_name'
]


# --- Step 2: Chain all transformations for the final DataFrame ---

df_display_ad = display_ad_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("actual_cpm", col("metrics_active_view_cpm") / 1000000) \
    .withColumn("total_cost", (col("metrics_active_view_impressions") / 1000) * col("actual_cpm")) \
    .select(
        "segments_date",
        "campaign_name",
        "ad_group_name",
        "metrics_active_view_impressions",
        col("metrics_active_view_measurability").alias("measurability_pct"),
        col("metrics_active_view_viewability").alias("viewability_pct"),
        col("actual_cpm").alias("cpm"),
        "total_cost"
    )

# --- Step 3: Display the final, cleaned output ---

print("--- Final Display Ad Schema ---")
df_display_ad.printSchema()

print("\n--- Final Transformed Display Ad Data ---")
display(df_display_ad.limit(10))

--- Final Display Ad Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- ad_group_name: string (nullable = false)
 |-- metrics_active_view_impressions: integer (nullable = false)
 |-- measurability_pct: double (nullable = false)
 |-- viewability_pct: double (nullable = false)
 |-- cpm: double (nullable = true)
 |-- total_cost: double (nullable = true)


--- Final Transformed Display Ad Data ---


segments_date,campaign_name,ad_group_name,metrics_active_view_impressions,measurability_pct,viewability_pct,cpm,total_cost
2024-10-04,Azure Data Engineering -Display - Image,Ad group 1,4691,0.9887477897444141,0.7626402211022598,65.30464421232146,306.34408599999995
2024-10-05,Azure Data Engineering -Display - Image,Ad group 1,2849,0.9697300245432233,0.8011811023622047,76.63080414180415,218.32116100000005
2024-10-06,Azure Data Engineering -Display - Image,Ad group 1,4482,0.976249760582264,0.8793407886992348,89.06343016510486,399.182294
2024-10-07,Azure Data Engineering -Display - Image,Ad group 1,7101,0.947713196760362,0.8924217669976122,42.053037037037036,298.618616
2024-10-08,Azure Data Engineering -Display - Image,Ad group 1,3892,0.9014560033762398,0.9110486891385767,32.180010791366904,125.24460199999999
2024-10-09,Azure Data Engineering -Display - Image,Ad group 1,10114,0.9736196319018405,0.9104329822666306,48.80586671939885,493.622536
2024-10-10,Azure Data Engineering -Display - Image,Ad group 1,7938,0.9663955479452054,0.8790697674418605,43.25428521038045,343.35251600000004
2024-10-11,Azure Data Engineering -Display - Image,Ad group 1,6431,0.960950080515298,0.8980589303169948,53.756026278961286,345.705005
2024-10-12,Azure Data Engineering -Display - Image,Ad group 1,5010,0.707092980521867,0.867983367983368,61.79834890219561,309.609728
2024-10-13,Azure Data Engineering -Display - Image,Ad group 1,2554,0.34552749586971915,0.8722677595628415,87.80658144087705,224.258009


In [0]:
"""
Displays the first 3 rows and schema of the call lead generation DataFrame for inspection.
"""
display(call_lead_generation_df.limit(3))
call_lead_generation_df.printSchema()

segments.date,campaign.name,ad_group.name,metrics.phone_calls,metrics.phone_impressions,metrics.phone_through_rate
2025-12-20,Azure Data Engineering -Display - Image,Ad group 1,0,15449,0.0
2025-12-21,Azure Data Engineering -Display - Image,Ad group 1,0,1792,0.0
2025-12-22,Azure Data Engineering -Display - Image,Ad group 1,0,22315,0.0


root
 |-- segments.date: date (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- ad_group.name: string (nullable = true)
 |-- metrics.phone_calls: integer (nullable = true)
 |-- metrics.phone_impressions: integer (nullable = true)
 |-- metrics.phone_through_rate: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning ---

# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in call_lead_generation_df.columns]

# Define lists for handling nulls using the NEW, cleaned names
numeric_cols_to_fill = [
    'metrics_phone_calls',
    'metrics_phone_impressions',
    'metrics_phone_through_rate'
]
string_cols_to_fill = [
    'campaign_name',
    'ad_group_name'
]


# --- Step 2: Chain all transformations for the final DataFrame ---

df_call_lead_generation = call_lead_generation_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("phone_through_rate_pct", col("metrics_phone_through_rate") * 100) \
    .select(
        "segments_date",
        "campaign_name",
        "ad_group_name",
        "metrics_phone_impressions",
        "metrics_phone_calls",
        col("phone_through_rate_pct").alias("phone_through_rate_pct")
    )


# --- Step 3: Display the final, cleaned output ---

print("--- Final Call Lead Schema ---")
df_call_lead_generation.printSchema()

print("\n--- Final Transformed Call Lead Data ---")
display(df_call_lead_generation.limit(10))

--- Final Call Lead Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- ad_group_name: string (nullable = false)
 |-- metrics_phone_impressions: integer (nullable = false)
 |-- metrics_phone_calls: integer (nullable = false)
 |-- phone_through_rate_pct: double (nullable = false)


--- Final Transformed Call Lead Data ---


segments_date,campaign_name,ad_group_name,metrics_phone_impressions,metrics_phone_calls,phone_through_rate_pct
2025-12-20,Azure Data Engineering -Display - Image,Ad group 1,15449,0,0.0
2025-12-21,Azure Data Engineering -Display - Image,Ad group 1,1792,0,0.0
2025-12-22,Azure Data Engineering -Display - Image,Ad group 1,22315,0,0.0
2024-10-04,Azure Data Engineering -Display - text,Ad group 1,176,0,0.0
2024-10-05,Azure Data Engineering -Display - text,Ad group 1,154,0,0.0
2024-10-06,Azure Data Engineering -Display - text,Ad group 1,216,0,0.0
2024-10-07,Azure Data Engineering -Display - text,Ad group 1,76,0,0.0
2024-10-07,Azure Data Engineering -Display - text,Ad group 2,1,0,0.0
2024-10-08,Azure Data Engineering -Display - text,Ad group 1,3,0,0.0
2024-10-08,Azure Data Engineering -Display - text,Ad group 2,2,0,0.0


In [0]:
geo_target_constant_df
"""
Displays the first 3 rows and schema of the geo target constant DataFrame for inspection.
"""
display(geo_target_constant_df.limit(3))
geo_target_constant_df.printSchema()

geo_target_constant.id,geo_target_constant.name,geo_target_constant.canonical_name,geo_target_constant.country_code,geo_target_constant.target_type
9142303,33634,"33634,Chihuahua,Mexico",MX,Postal Code
9142304,77527,"77527,Quintana Roo,Mexico",MX,Postal Code
9142305,40867,"40867,Guerrero,Mexico",MX,Postal Code


root
 |-- geo_target_constant.id: string (nullable = true)
 |-- geo_target_constant.name: string (nullable = true)
 |-- geo_target_constant.canonical_name: string (nullable = true)
 |-- geo_target_constant.country_code: string (nullable = true)
 |-- geo_target_constant.target_type: string (nullable = true)



In [0]:
"""
Geo target constant reference data.
Acts as a lookup table for location IDs found in other tables.
"""
# --- Step 1: Define new names and column lists for cleaning ---
new_column_names = [c.replace('.', '_') for c in geo_target_constant_df.columns]

# We ensure IDs are treated as strings to handle 64-bit precision safety.
numeric_cols_to_fill = [] 

string_cols_to_fill = [
    'geo_target_constant_id',
    'geo_target_constant_name',
    'geo_target_constant_canonical_name',
    'geo_target_constant_country_code',
    'geo_target_constant_target_type'
]

# --- Step 2: Chain all transformations for the final DataFrame ---
df_geo_target_constant = geo_target_constant_df.toDF(*new_column_names) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .select(
        col("geo_target_constant_id").alias("constant_id"),
        col("geo_target_constant_name").alias("name"),
        col("geo_target_constant_canonical_name").alias("canonical_name"),
        col("geo_target_constant_country_code").alias("country_code"),
        col("geo_target_constant_target_type").alias("target_type")
    )

# --- Step 3: Display the final, cleaned output ---
print("--- Final Geo Target Constant Schema ---")
df_geo_target_constant.printSchema()

print("\n--- Final Transformed Geo Target Constant Data ---")
display(df_geo_target_constant.limit(10))

--- Final Geo Target Constant Schema ---
root
 |-- constant_id: string (nullable = false)
 |-- name: string (nullable = false)
 |-- canonical_name: string (nullable = false)
 |-- country_code: string (nullable = false)
 |-- target_type: string (nullable = false)


--- Final Transformed Geo Target Constant Data ---


constant_id,name,canonical_name,country_code,target_type
9142303,33634,"33634,Chihuahua,Mexico",MX,Postal Code
9142304,77527,"77527,Quintana Roo,Mexico",MX,Postal Code
9142305,40867,"40867,Guerrero,Mexico",MX,Postal Code
9142306,33157,"33157,Chihuahua,Mexico",MX,Postal Code
9142307,80720,"80720,Sinaloa,Mexico",MX,Postal Code
9142308,35930,"35930,Durango,Mexico",MX,Postal Code
9142309,92917,"92917,Veracruz,Mexico",MX,Postal Code
9142310,73764,"73764,Puebla,Mexico",MX,Postal Code
9142311,37665,"37665,Guanajuato,Mexico",MX,Postal Code
9142312,30407,"30407,Chiapas,Mexico",MX,Postal Code


In [0]:
"""
Displays the first 3 rows and schema of the geographic performance DataFrame for inspection.
"""
display(geographic_df.limit(3))
geographic_df.printSchema()

campaign.id,segments.date,geographic_view.country_criterion_id,segments.geo_target_city,geographic_view.location_type,metrics.impressions,metrics.clicks,metrics.conversions,metrics.cost_micros,metrics.ctr
21519858184,2024-09-03,2356,geoTargetConstants/9300190,LOCATION_OF_PRESENCE,1,0,0.0,0.0,0.0
21519858184,2024-09-03,2356,geoTargetConstants/9300288,LOCATION_OF_PRESENCE,1,0,0.0,0.0,0.0
21519858184,2024-09-03,2356,geoTargetConstants/9300294,LOCATION_OF_PRESENCE,11,0,0.0,0.0,0.0


root
 |-- campaign.id: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- geographic_view.country_criterion_id: string (nullable = true)
 |-- segments.geo_target_city: string (nullable = true)
 |-- geographic_view.location_type: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.conversions: double (nullable = true)
 |-- metrics.cost_micros: double (nullable = true)
 |-- metrics.ctr: double (nullable = true)



In [0]:
# --- Step 1: Define new names and column lists for cleaning ---

# Generate new column names by replacing '.' with '_'
new_column_names = [c.replace('.', '_') for c in geographic_df.columns]

# Define lists for handling nulls using the NEW, cleaned names
numeric_cols_to_fill = [
    'metrics_impressions',
    'metrics_clicks',
    'metrics_conversions',
    'metrics_cost_micros'
]
string_cols_to_fill = [
    'segments_geo_target_city',
    'geographic_view_location_type'
]


# --- Step 2: Chain all transformations for the final DataFrame ---

df_geographic = geographic_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn(
        "cost_inr",
        col("metrics_cost_micros") / 1000000
    ) \
    .withColumn("ctr_pct",
        when(col("metrics_impressions") > 0, (col("metrics_clicks") / col("metrics_impressions")) * 100)
        .otherwise(0)
    ) \
    .withColumn("cvr_pct",
        when(col("metrics_clicks") > 0, (col("metrics_conversions") / col("metrics_clicks")) * 100)
        .otherwise(0)
    ) \
    .withColumn("city_id", regexp_extract(col("segments_geo_target_city"), r'/(\d+)$', 1)) \
    .select(
        "segments_date",
        "campaign_id",
        "city_id",
        col("geographic_view_location_type").alias("location_type"),
        "metrics_impressions",
        "metrics_clicks",
        "ctr_pct",
        "metrics_conversions",
        "cvr_pct",
        "cost_inr"

)

# --- Step 3: Display the final, cleaned output ---

print("--- Final Geographic Schema ---")
df_geographic.printSchema()

print("\n--- Final Transformed Geographic Data ---")
display(df_geographic.limit(10))

--- Final Geographic Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- city_id: string (nullable = false)
 |-- location_type: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- ctr_pct: double (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cvr_pct: double (nullable = true)
 |-- cost_inr: double (nullable = true)


--- Final Transformed Geographic Data ---


segments_date,campaign_id,city_id,location_type,metrics_impressions,metrics_clicks,ctr_pct,metrics_conversions,cvr_pct,cost_inr
2025-12-22,21771245709,1007785,AREA_OF_INTEREST,9645,1430,14.826334888543286,0.0,0.0,478.195495
2025-02-06,21771245709,1007785,AREA_OF_INTEREST,10084,1176,11.662038873462912,0.0,0.0,179.60372
2025-02-11,21771245709,1007785,AREA_OF_INTEREST,7170,1075,14.99302649930265,0.0,0.0,131.52735
2025-02-10,21771245709,1007785,AREA_OF_INTEREST,6815,941,13.807776962582539,0.0,0.0,143.134576
2025-02-07,21771245709,1007785,AREA_OF_INTEREST,6675,890,13.333333333333334,0.0,0.0,172.378663
2025-02-08,21771245709,1007785,AREA_OF_INTEREST,4962,645,12.998790810157196,0.0,0.0,151.495722
2025-02-09,21771245709,1007785,AREA_OF_INTEREST,4297,536,12.473818943448919,0.0,0.0,109.456635
2025-12-20,21771245709,1007785,AREA_OF_INTEREST,4871,505,10.367481010059535,0.0,0.0,160.251089
2024-11-08,21771245709,1007768,AREA_OF_INTEREST,3132,463,14.782886334610474,0.0,0.0,262.520931
2025-02-05,21771245709,1007785,AREA_OF_INTEREST,4788,447,9.335839598997493,0.0,0.0,169.111592


In [0]:
df_geographic_enriched = df_geographic.join(
    df_geo_target_constant,
    df_geographic.city_id == df_geo_target_constant.constant_id,
    "left"
).select(
    df_geographic["*"],
    df_geo_target_constant["name"].alias("city_name"),
    df_geo_target_constant["canonical_name"],
    df_geo_target_constant["country_code"]
)

print("--- Final Schema ---")
df_geographic_enriched.printSchema()

print("\n--- Final Transformed Data ---")
display(df_geographic_enriched.limit(10))

--- Final Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- city_id: string (nullable = false)
 |-- location_type: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- ctr_pct: double (nullable = true)
 |-- metrics_conversions: double (nullable = false)
 |-- cvr_pct: double (nullable = true)
 |-- cost_inr: double (nullable = true)
 |-- city_name: string (nullable = true)
 |-- canonical_name: string (nullable = true)
 |-- country_code: string (nullable = true)


--- Final Transformed Data ---


segments_date,campaign_id,city_id,location_type,metrics_impressions,metrics_clicks,ctr_pct,metrics_conversions,cvr_pct,cost_inr,city_name,canonical_name,country_code
2025-12-22,21771245709,1007785,AREA_OF_INTEREST,9645,1430,14.826334888543286,0.0,0.0,478.195495,Mumbai,"Mumbai,Maharashtra,India",IN
2025-02-06,21771245709,1007785,AREA_OF_INTEREST,10084,1176,11.662038873462912,0.0,0.0,179.60372,Mumbai,"Mumbai,Maharashtra,India",IN
2025-02-11,21771245709,1007785,AREA_OF_INTEREST,7170,1075,14.99302649930265,0.0,0.0,131.52735,Mumbai,"Mumbai,Maharashtra,India",IN
2025-02-10,21771245709,1007785,AREA_OF_INTEREST,6815,941,13.807776962582539,0.0,0.0,143.134576,Mumbai,"Mumbai,Maharashtra,India",IN
2025-02-07,21771245709,1007785,AREA_OF_INTEREST,6675,890,13.333333333333334,0.0,0.0,172.378663,Mumbai,"Mumbai,Maharashtra,India",IN
2025-02-08,21771245709,1007785,AREA_OF_INTEREST,4962,645,12.998790810157196,0.0,0.0,151.495722,Mumbai,"Mumbai,Maharashtra,India",IN
2025-02-09,21771245709,1007785,AREA_OF_INTEREST,4297,536,12.473818943448919,0.0,0.0,109.456635,Mumbai,"Mumbai,Maharashtra,India",IN
2025-12-20,21771245709,1007785,AREA_OF_INTEREST,4871,505,10.367481010059535,0.0,0.0,160.251089,Mumbai,"Mumbai,Maharashtra,India",IN
2024-11-08,21771245709,1007768,AREA_OF_INTEREST,3132,463,14.782886334610474,0.0,0.0,262.520931,Bengaluru,"Bengaluru,Karnataka,India",IN
2025-02-05,21771245709,1007785,AREA_OF_INTEREST,4788,447,9.335839598997493,0.0,0.0,169.111592,Mumbai,"Mumbai,Maharashtra,India",IN


In [0]:
"""
Displays the first 3 rows and schema of the ads performance DataFrame for inspection.
"""
display(ads_performance_df.limit(3))
ads_performance_df.printSchema()


ad_group.id,ad_group_ad.ad.id,metrics.impressions,metrics.clicks,metrics.conversions,metrics.cost_micros
169993022084,715897121402,691883,66109,0.0,21791049587
169807400833,715777857354,6110,209,2.0,9534278957
169807400833,720868651374,13926,189,1.0,10521844824


root
 |-- ad_group.id: string (nullable = true)
 |-- ad_group_ad.ad.id: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.conversions: double (nullable = true)
 |-- metrics.cost_micros: long (nullable = true)



In [0]:
new_column_names = [c.replace('.', '_') for c in ads_performance_df.columns]

df_ads_performance = ads_performance_df.toDF(*new_column_names)

#  Define null handling lists
numeric_cols_to_fill = [
    'metrics_impressions', 'metrics_clicks',
    'metrics_conversions', 'metrics_cost_micros'
]
string_cols_to_fill = [
    'ad_group_id', 'ad_group_ad_ad_id'
]

#  Apply transformations
df_ads_performance = ads_performance_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("cost_inr", col("metrics_cost_micros") / 1000000) \
    .select(
        "ad_group_id",
        "ad_group_ad_ad_id",
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        "cost_inr"
    )

print("--- Ads Performance Schema ---")
df_ads_performance.printSchema()
print("\n--- Ads Performance Sample ---")
display(df_ads_performance.limit(10))


--- Ads Performance Schema ---
root
 |-- ad_group_id: string (nullable = false)
 |-- ad_group_ad_ad_id: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- metrics_conversions: double (nullable = false)
 |-- cost_inr: double (nullable = true)


--- Ads Performance Sample ---


ad_group_id,ad_group_ad_ad_id,metrics_impressions,metrics_clicks,metrics_conversions,cost_inr
169993022084,715897121402,691883,66109,0.0,21791.049587
169807400833,715777857354,6110,209,2.0,9534.278957
169807400833,720868651374,13926,189,1.0,10521.844824
169807400833,734702302019,0,0,0.0,0.0
169808622633,716160811571,4,0,0.0,0.0
175636335299,734387732622,5838,170,0.0,3074.541199
177495674774,749552696579,244220,10782,4.0,33294.98356
185910449435,773303116932,53903,444,0.0,18755.628475
192135530842,789417704773,12585,525,132.0,15509.086305


In [0]:
"""
Displays the first 3 rows and schema of the ads data DataFrame for inspection.
"""
display(ads_data_df.limit(3))
ads_data_df.printSchema()

campaign.id,campaign.name,ad_group.id,ad_group.name,ad_group_ad.ad.id,ad_group_ad.ad.type,ad_group_ad.status,ad_group_ad.ad.final_urls,ad_group_ad.ad.responsive_search_ad.headlines,ad_group_ad.ad.responsive_search_ad.descriptions,ad_group_ad.ad.expanded_text_ad.headline_part1,ad_group_ad.ad.expanded_text_ad.headline_part2,ad_group_ad.ad.expanded_text_ad.headline_part3,ad_group_ad.ad.expanded_text_ad.description,ad_group_ad.ad.expanded_text_ad.description2,ad_group_ad.ad.responsive_display_ad.long_headline,ad_group_ad.ad.responsive_display_ad.headlines,ad_group_ad.ad.responsive_display_ad.descriptions,ad_group_ad.ad.image_ad.image_url,ad_group_ad.ad.image_ad.mime_type
21771245709,Azure Data Engineering -Display - Image,169993022084,Ad group 1,715897121402,RESPONSIVE_DISPLAY_AD,PAUSED,https://academyofdata.ai,null,null,null,null,null,null,null,null,"text: ""Data Engineering Training"" , text: ""Azure Data Engineering Course"" , text: ""Best Data Engineering Program"" , text: ""Learn Data Engineering"" , text: ""Data Engineering Certification""","text: ""Master cloud data engineering and become a certified Azure professional."" , text: ""Flexible timings, expert trainers. Placement assistance. Limited seats - Join now!"" , text: ""Upskill with Azure Data Engineering training and unlock top tech career opportunities."" , text: ""Gain hands-on Azure skills to excel in high-demand data engineering roles."" , text: ""Discover the Academy of Data, Your Gateway to Mastering the Art of Data Engineering""",null,null
21781297084,Azure Data Engineering -Display - text,169807400833,Ad group 1,715777857354,CALL_AD,REMOVED,https://academyofdata.in/,null,null,null,null,null,null,null,null,null,null,null,null
21781297084,Azure Data Engineering -Display - text,169807400833,Ad group 1,720868651374,CALL_AD,REMOVED,https://academyofdata.in/,null,null,null,null,null,null,null,null,null,null,null,null


root
 |-- campaign.id: string (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- ad_group.id: string (nullable = true)
 |-- ad_group.name: string (nullable = true)
 |-- ad_group_ad.ad.id: string (nullable = true)
 |-- ad_group_ad.ad.type: string (nullable = true)
 |-- ad_group_ad.status: string (nullable = true)
 |-- ad_group_ad.ad.final_urls: string (nullable = true)
 |-- ad_group_ad.ad.responsive_search_ad.headlines: string (nullable = true)
 |-- ad_group_ad.ad.responsive_search_ad.descriptions: string (nullable = true)
 |-- ad_group_ad.ad.expanded_text_ad.headline_part1: string (nullable = true)
 |-- ad_group_ad.ad.expanded_text_ad.headline_part2: string (nullable = true)
 |-- ad_group_ad.ad.expanded_text_ad.headline_part3: string (nullable = true)
 |-- ad_group_ad.ad.expanded_text_ad.description: string (nullable = true)
 |-- ad_group_ad.ad.expanded_text_ad.description2: string (nullable = true)
 |-- ad_group_ad.ad.responsive_display_ad.long_headline: string (nulla

In [0]:
# Step 1: Rename all columns by replacing '.' with '_'
# This step remains the same and correctly handles all columns.
new_column_names = [c.replace('.', '_') for c in ads_data_df.columns]
df_ads_data = ads_data_df.toDF(*new_column_names)

# Step 2: Define lists for handling null values based on the new schema
# We'll handle string and numeric types separately for correctness.

string_cols_to_fill = [
    'campaign_id', 'campaign_name', 
    'ad_group_id', 'ad_group_name', 
    'ad_group_ad_ad_id', 'ad_group_ad_ad_type', 'ad_group_ad_status', 
    'ad_group_ad_ad_final_urls', 
    'ad_group_ad_ad_responsive_search_ad_headlines', 
    'ad_group_ad_ad_responsive_search_ad_descriptions', 
    'ad_group_ad_ad_expanded_text_ad_headline_part1', 
    'ad_group_ad_ad_expanded_text_ad_headline_part2', 
    'ad_group_ad_ad_expanded_text_ad_headline_part3', 
    'ad_group_ad_ad_expanded_text_ad_description',
    'ad_group_ad_ad_expanded_text_ad_description2',
    'ad_group_ad_ad_responsive_display_ad_long_headline', 
    'ad_group_ad_ad_responsive_display_ad_headlines',  
    'ad_group_ad_ad_responsive_display_ad_descriptions', 
    'ad_group_ad_ad_image_ad_image_url',
    'ad_group_ad_ad_image_ad_mime_type'
    ##deprecated columns from googleadsv23
    #'ad_group_ad_ad_call_ad_business_name', 
    #'ad_group_ad_ad_call_ad_country_code', 
    #'ad_group_ad_ad_call_ad_phone_number', 
    #'ad_group_ad_ad_call_ad_headline1', 
    #'ad_group_ad_ad_call_ad_headline2', 
    #'ad_group_ad_ad_call_ad_description1', 
    #'ad_group_ad_ad_call_ad_description2', 
]

# Step 3: Apply transformations
# Fill nulls and keep all columns (the .select() is removed).

df_ads_data = ads_data_df.toDF(*new_column_names) \
    .fillna('Unknown', subset=string_cols_to_fill)

# --- Verification ---
print("--- Ads Data Schema ---")
df_ads_data.printSchema()
print("\n--- Ads Data Sample ---")
display(df_ads_data.limit(10))

--- Ads Data Schema ---
root
 |-- campaign_id: string (nullable = false)
 |-- campaign_name: string (nullable = false)
 |-- ad_group_id: string (nullable = false)
 |-- ad_group_name: string (nullable = false)
 |-- ad_group_ad_ad_id: string (nullable = false)
 |-- ad_group_ad_ad_type: string (nullable = false)
 |-- ad_group_ad_status: string (nullable = false)
 |-- ad_group_ad_ad_final_urls: string (nullable = false)
 |-- ad_group_ad_ad_responsive_search_ad_headlines: string (nullable = false)
 |-- ad_group_ad_ad_responsive_search_ad_descriptions: string (nullable = false)
 |-- ad_group_ad_ad_expanded_text_ad_headline_part1: string (nullable = false)
 |-- ad_group_ad_ad_expanded_text_ad_headline_part2: string (nullable = false)
 |-- ad_group_ad_ad_expanded_text_ad_headline_part3: string (nullable = false)
 |-- ad_group_ad_ad_expanded_text_ad_description: string (nullable = false)
 |-- ad_group_ad_ad_expanded_text_ad_description2: string (nullable = false)
 |-- ad_group_ad_ad_responsive_

campaign_id,campaign_name,ad_group_id,ad_group_name,ad_group_ad_ad_id,ad_group_ad_ad_type,ad_group_ad_status,ad_group_ad_ad_final_urls,ad_group_ad_ad_responsive_search_ad_headlines,ad_group_ad_ad_responsive_search_ad_descriptions,ad_group_ad_ad_expanded_text_ad_headline_part1,ad_group_ad_ad_expanded_text_ad_headline_part2,ad_group_ad_ad_expanded_text_ad_headline_part3,ad_group_ad_ad_expanded_text_ad_description,ad_group_ad_ad_expanded_text_ad_description2,ad_group_ad_ad_responsive_display_ad_long_headline,ad_group_ad_ad_responsive_display_ad_headlines,ad_group_ad_ad_responsive_display_ad_descriptions,ad_group_ad_ad_image_ad_image_url,ad_group_ad_ad_image_ad_mime_type
21771245709,Azure Data Engineering -Display - Image,169993022084,Ad group 1,715897121402,RESPONSIVE_DISPLAY_AD,PAUSED,https://academyofdata.ai,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,"text: ""Data Engineering Training"" , text: ""Azure Data Engineering Course"" , text: ""Best Data Engineering Program"" , text: ""Learn Data Engineering"" , text: ""Data Engineering Certification""","text: ""Master cloud data engineering and become a certified Azure professional."" , text: ""Flexible timings, expert trainers. Placement assistance. Limited seats - Join now!"" , text: ""Upskill with Azure Data Engineering training and unlock top tech career opportunities."" , text: ""Gain hands-on Azure skills to excel in high-demand data engineering roles."" , text: ""Discover the Academy of Data, Your Gateway to Mastering the Art of Data Engineering""",Unknown,Unknown
21781297084,Azure Data Engineering -Display - text,169807400833,Ad group 1,715777857354,CALL_AD,REMOVED,https://academyofdata.in/,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
21781297084,Azure Data Engineering -Display - text,169807400833,Ad group 1,720868651374,CALL_AD,REMOVED,https://academyofdata.in/,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
21781297084,Azure Data Engineering -Display - text,169807400833,Ad group 1,734702302019,CALL_AD,ENABLED,https://academyofdata.ai/,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
21781297084,Azure Data Engineering -Display - text,169808622633,Ad group 2,716160811571,RESPONSIVE_SEARCH_AD,ENABLED,https://academyofdata.in/,"text: ""best data engineering courses"" asset_performance_label: PENDING policy_summary_info { review_status: REVIEWED approval_status: APPROVED } , text: ""Data science course near me"" asset_performance_label: PENDING policy_summary_info { review_status: REVIEWED approval_status: APPROVED } , text: ""Data Engineering Certification"" asset_performance_label: PENDING policy_summary_info { review_status: REVIEWED approval_status: APPROVED } , text: ""Azure Data Engineering Course"" asset_performance_label: PENDING policy_summary_info { review_status: REVIEWED approval_status: APPROVED } , text: ""Learn Data Engineering Today"" asset_performance_label: PENDING policy_summary_info { review_status: REVIEWED approval_status: APPROVED } , text: ""Data Engineering Training"" asset_performance_label: PENDING policy_summary_info { review_status: REVIEWED approval_status: APPROVED } , text: ""Discover AWS & Snowflake"" asset_performance_label: PENDING policy_summary_info { review_status: REVIEWED approval_status: APPROVED } , text: ""Future of Data Engineering"" asset_performance_label: PENDING policy_summary_info { review_status: REVIEWED approval_status: APPROVED } , text: ""Training Programs"" asset_performance_label: PENDING policy_summary_info { review_status: REVIEWED approval_status: APPROVED } , text: ""Academy of Data"" asset_performance_label: PENDING policy_summary_info { review_status: REVIEWED approval_status: APPROVED } , text: ""Data Science Course"" asset_performance_label: PENDING policy_summary_info { review_status: REVIEWED approval_status: APPROVED } , text: ""Best Data Engineerin

In [0]:
for a in df_ads_data.columns:
    print(a)

campaign_id
campaign_name
ad_group_id
ad_group_name
ad_group_ad_ad_id
ad_group_ad_ad_type
ad_group_ad_status
ad_group_ad_ad_final_urls
ad_group_ad_ad_responsive_search_ad_headlines
ad_group_ad_ad_responsive_search_ad_descriptions
ad_group_ad_ad_expanded_text_ad_headline_part1
ad_group_ad_ad_expanded_text_ad_headline_part2
ad_group_ad_ad_expanded_text_ad_headline_part3
ad_group_ad_ad_expanded_text_ad_description
ad_group_ad_ad_expanded_text_ad_description2
ad_group_ad_ad_responsive_display_ad_long_headline
ad_group_ad_ad_responsive_display_ad_headlines
ad_group_ad_ad_responsive_display_ad_descriptions
ad_group_ad_ad_image_ad_image_url
ad_group_ad_ad_image_ad_mime_type


In [0]:
"""
Displays the first 3 rows and schema of the campaign asset DataFrame for inspection.
"""
display(campaign_asset_df.limit(3))
campaign_asset_df.printSchema()

campaign.id,campaign.name,campaign.status,asset.id,asset.type,campaign_asset.status,segments.date,segments.device,metrics.impressions,metrics.clicks,metrics.cost_micros,metrics.conversions,metrics.ctr,metrics.average_cpc
22973632076,Data Engineering (AWS-Sept),PAUSED,231020009021,TEXT,ENABLED,2025-09-08,MOBILE,31257,1412,1.317624832E9,0.0,0.045173881050644654,933162.0623229461
22973632076,Data Engineering (AWS-Sept),PAUSED,283767982563,IMAGE,ENABLED,2025-09-08,MOBILE,31136,1395,9.74979618E8,0.0,0.04480344295991778,698910.1204301076
21771245709,Azure Data Engineering -Display - Image,PAUSED,176383668253,CALL,ENABLED,2025-12-22,MOBILE,22515,3041,9.82748377E8,0.0,0.13506551188096824,323166.18776718184


root
 |-- campaign.id: string (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- campaign.status: string (nullable = true)
 |-- asset.id: string (nullable = true)
 |-- asset.type: string (nullable = true)
 |-- campaign_asset.status: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- segments.device: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.cost_micros: double (nullable = true)
 |-- metrics.conversions: double (nullable = true)
 |-- metrics.ctr: double (nullable = true)
 |-- metrics.average_cpc: double (nullable = true)



In [0]:
"""
Campaign-level asset performance
"""
new_column_names = [c.replace('.', '_') for c in campaign_asset_df.columns]

numeric_cols_to_fill = [
    'metrics_impressions', 'metrics_clicks', 'metrics_cost_micros',
    'metrics_conversions', 'metrics_ctr', 'metrics_average_cpc'
]
string_cols_to_fill = [
    'campaign_name', 'campaign_status', 'asset_type',
    'campaign_asset_status', 'segments_device'
]

df_campaign_asset = campaign_asset_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("ctr_pct", col("metrics_ctr") * 100) \
    .withColumn("cost_inr", col("metrics_cost_micros") / 1000000) \
    .withColumn("cpc_inr", col("metrics_average_cpc") / 1000000) \
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        "campaign_status",
        "asset_id",
        "asset_type",
        col("campaign_asset_status").alias("asset_status"),
        col("segments_device").alias("device"),
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        "ctr_pct",
        "cpc_inr",
        "cost_inr"
    )

print("--- Campaign Asset Schema ---")
df_campaign_asset.printSchema()
display(df_campaign_asset.limit(10))

--- Campaign Asset Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- campaign_status: string (nullable = false)
 |-- asset_id: string (nullable = true)
 |-- asset_type: string (nullable = false)
 |-- asset_status: string (nullable = false)
 |-- device: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- metrics_conversions: double (nullable = false)
 |-- ctr_pct: double (nullable = false)
 |-- cpc_inr: double (nullable = true)
 |-- cost_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,campaign_status,asset_id,asset_type,asset_status,device,metrics_impressions,metrics_clicks,metrics_conversions,ctr_pct,cpc_inr,cost_inr
2025-09-08,22973632076,Data Engineering (AWS-Sept),PAUSED,231020009021,TEXT,ENABLED,MOBILE,31257,1412,0.0,4.517388105064465,0.9331620623229462,1317.624832
2025-09-08,22973632076,Data Engineering (AWS-Sept),PAUSED,283767982563,IMAGE,ENABLED,MOBILE,31136,1395,0.0,4.480344295991778,0.6989101204301076,974.979618
2025-12-22,21771245709,Azure Data Engineering -Display - Image,PAUSED,176383668253,CALL,ENABLED,MOBILE,22515,3041,0.0,13.506551188096823,0.3231661877671818,982.748377
2025-12-20,21771245709,Azure Data Engineering -Display - Image,PAUSED,176383668253,CALL,ENABLED,MOBILE,15872,1340,0.0,8.442540322580646,0.34303179925373134,459.662611
2024-08-02,21519858184,Excel in Data Engineering,PAUSED,154199286142,TEXT,ENABLED,MOBILE,14359,581,0.0,4.046242774566474,0.3739309690189329,217.253893
2025-09-05,22973632076,Data Engineering (AWS-Sept),PAUSED,204178822909,SITELINK,ENABLED,MOBILE,13967,1632,0.0,11.684685329705735,0.6858854895833334,1119.365119
2025-09-05,22973632076,Data Engineering (AWS-Sept),PAUSED,231034323696,SITELINK,ENABLED,MOBILE,13967,1632,0.0,11.684685329705735,0.6858854895833334,1119.365119
2025-09-05,22973632076,Data Engineering (AWS-Sept),PAUSED,204178822912,SITELINK,ENABLED,MOBILE,13965,1632,0.0,11.686358754027927,0.6858854895833334,1119.365119
2025-07-09,22489991406,Hyderabad,PAUSED,231020009021,TEXT,ENABLED,MOBILE,13915,393,0.0,2.824290334171757,1.8080318804071247,710.556529
2024-08-02,21519858184,Excel in Data Engineering,PAUSED,153986125882,IMAGE,ENABLED,MOBILE,13708,545,0.0,3.9757805660927925,0.3744273100917431,204.062884


In [0]:
"""
Displays the first 3 rows and schema of the ad group ad asset DataFrame for inspection.
"""
display(ad_group_ad_asset_view_df.limit(3))
ad_group_ad_asset_view_df.printSchema()

campaign.id,campaign.name,ad_group.id,ad_group.name,ad_group_ad_asset_view.asset,ad_group_ad_asset_view.field_type,segments.date,segments.device,metrics.impressions,metrics.clicks,metrics.cost_micros,metrics.conversions,metrics.ctr,metrics.average_cpc
22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,customers/1401815809/assets/286842315788,HEADLINE,2025-09-18,DESKTOP,166,1,1.0871E8,0.0,0.006024096385542169,1.0871E8
22489991406,Hyderabad,177495674774,Ad group 1,customers/1401815809/assets/231102804790,HEADLINE,2025-05-13,DESKTOP,165,0,0.0,0.0,0.0,null
22489991406,Hyderabad,177495674774,Ad group 1,customers/1401815809/assets/231102804814,HEADLINE,2025-07-10,MOBILE,165,5,9840651.0,0.0,0.030303030303030304,1968130.2


root
 |-- campaign.id: string (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- ad_group.id: string (nullable = true)
 |-- ad_group.name: string (nullable = true)
 |-- ad_group_ad_asset_view.asset: string (nullable = true)
 |-- ad_group_ad_asset_view.field_type: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- segments.device: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.cost_micros: double (nullable = true)
 |-- metrics.conversions: double (nullable = true)
 |-- metrics.ctr: double (nullable = true)
 |-- metrics.average_cpc: double (nullable = true)



In [0]:
"""
Ad group ad-level asset performance
"""
new_column_names = [c.replace('.', '_') for c in ad_group_ad_asset_view_df.columns]

numeric_cols_to_fill = [
    'metrics_impressions', 'metrics_clicks', 'metrics_cost_micros',
    'metrics_conversions', 'metrics_ctr', 'metrics_average_cpc'
]
string_cols_to_fill = [
    'campaign_name', 'ad_group_name', 'ad_group_ad_asset_view_asset',
    'ad_group_ad_asset_view_field_type', 'segments_device'
]

df_ad_group_ad_asset_view = ad_group_ad_asset_view_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("ctr_pct", col("metrics_ctr") * 100) \
    .withColumn("cost_inr", col("metrics_cost_micros") / 1000000) \
    .withColumn("cpc_inr", col("metrics_average_cpc") / 1000000) \
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        "ad_group_id",
        "ad_group_name",
        col("ad_group_ad_asset_view_asset").alias("asset"),
        col("ad_group_ad_asset_view_field_type").alias("field_type"),
        col("segments_device").alias("device"),
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        "ctr_pct",
        "cpc_inr",
        "cost_inr"
    )

print("--- Ad Group Ad Asset View Schema ---")
df_ad_group_ad_asset_view.printSchema()
display(df_ad_group_ad_asset_view.limit(10))

--- Ad Group Ad Asset View Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- ad_group_id: string (nullable = true)
 |-- ad_group_name: string (nullable = false)
 |-- asset: string (nullable = false)
 |-- field_type: string (nullable = false)
 |-- device: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- metrics_conversions: double (nullable = false)
 |-- ctr_pct: double (nullable = false)
 |-- cpc_inr: double (nullable = true)
 |-- cost_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,ad_group_id,ad_group_name,asset,field_type,device,metrics_impressions,metrics_clicks,metrics_conversions,ctr_pct,cpc_inr,cost_inr
2025-09-18,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,customers/1401815809/assets/286842315788,HEADLINE,DESKTOP,166,1,0.0,0.6024096385542169,108.71,108.71
2025-05-13,22489991406,Hyderabad,177495674774,Ad group 1,customers/1401815809/assets/231102804790,HEADLINE,DESKTOP,165,0,0.0,0.0,0.0,0.0
2025-07-10,22489991406,Hyderabad,177495674774,Ad group 1,customers/1401815809/assets/231102804814,HEADLINE,MOBILE,165,5,0.0,3.0303030303030303,1.9681302,9.840651
2025-09-13,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,customers/1401815809/assets/286924483282,HEADLINE,DESKTOP,165,2,0.0,1.2121212121212122,34.095,68.19
2026-01-03,23393627674,Azure-Data-Engineering-Jan-2026,192135530842,Azure Data Engineering – Databricks,customers/1401815809/assets/176016713194,HEADLINE,MOBILE,165,8,2.0,4.848484848484849,28.73875,229.91
2025-05-03,22489991406,Hyderabad,177495674774,Ad group 1,customers/1401815809/assets/231102804817,HEADLINE,MOBILE,164,0,0.0,0.0,0.0,0.0
2025-05-08,22489991406,Hyderabad,177495674774,Ad group 1,customers/1401815809/assets/231547904897,HEADLINE,MOBILE,164,0,0.0,0.0,0.0,0.0
2025-09-11,22998241842,10-Sept AWS Snowflake,185910449435,Academy Of Data Engineering Course Registration Form,customers/1401815809/assets/286842315806,DESCRIPTION,DESKTOP,164,3,0.0,1.8292682926829267,153.65333333333334,460.96
2026-01-13,23393627674,Azure-Data-Engineering-Jan-2026,192135530842,Azure Data Engineering – Databricks,customers/1401815809/assets/318558464256,HEADLINE,MOBILE,164,8,6.0,4.878048780487805,26.995,215.96
2026-01-15,23393627674,Azure-Data-Engineering-Jan-2026,192135530842,Azure Data Engineering – Databricks,customers/1401815809/assets/318558464274,HEADLINE,MOBILE,164,8,3.0,4.878048780487805,31.0525,248.42


In [0]:
"""
Displays the first 3 rows and schema of the asset DataFrame for inspection.
"""
display(asset_df.limit(3))
asset_df.printSchema()

asset.id,asset.type,asset.source,asset.text_asset.text,asset.image_asset.full_size.url,asset.sitelink_asset.link_text,asset.callout_asset.callout_text,asset.structured_snippet_asset.header,asset.structured_snippet_asset.values
154024923624,TEXT,ADVERTISER,Excel in Data Engineering,null,null,null,null,null
154024923627,TEXT,ADVERTISER,Top Data Science Training,null,null,null,null,null
154024923630,TEXT,ADVERTISER,Get Azure & AWS Skills Now,null,null,null,null,null


root
 |-- asset.id: string (nullable = true)
 |-- asset.type: string (nullable = true)
 |-- asset.source: string (nullable = true)
 |-- asset.text_asset.text: string (nullable = true)
 |-- asset.image_asset.full_size.url: string (nullable = true)
 |-- asset.sitelink_asset.link_text: string (nullable = true)
 |-- asset.callout_asset.callout_text: string (nullable = true)
 |-- asset.structured_snippet_asset.header: string (nullable = true)
 |-- asset.structured_snippet_asset.values: string (nullable = true)



In [0]:
"""
Asset master data (text, images, sitelinks, callouts, structured snippets)
"""
new_column_names = [c.replace('.', '_') for c in asset_df.columns]

string_cols_to_fill = [
    'asset_type', 'asset_source', 'asset_text_asset_text',
    'asset_image_asset_full_size_url', 'asset_sitelink_asset_link_text',
    'asset_callout_asset_callout_text', 'asset_structured_snippet_asset_header',
    'asset_structured_snippet_asset_values'
]

df_asset = asset_df.toDF(*new_column_names) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .select(
        "asset_id",
        "asset_type",
        "asset_source",
        col("asset_text_asset_text").alias("text_asset_text"),
        col("asset_image_asset_full_size_url").alias("image_url"),
        col("asset_sitelink_asset_link_text").alias("sitelink_text"),
        col("asset_callout_asset_callout_text").alias("callout_text"),
        col("asset_structured_snippet_asset_header").alias("snippet_header"),
        col("asset_structured_snippet_asset_values").alias("snippet_values")
    )

print("--- Asset Schema ---")
df_asset.printSchema()
display(df_asset.limit(10))

--- Asset Schema ---
root
 |-- asset_id: string (nullable = true)
 |-- asset_type: string (nullable = false)
 |-- asset_source: string (nullable = false)
 |-- text_asset_text: string (nullable = false)
 |-- image_url: string (nullable = false)
 |-- sitelink_text: string (nullable = false)
 |-- callout_text: string (nullable = false)
 |-- snippet_header: string (nullable = false)
 |-- snippet_values: string (nullable = false)



asset_id,asset_type,asset_source,text_asset_text,image_url,sitelink_text,callout_text,snippet_header,snippet_values
154024923624,TEXT,ADVERTISER,Excel in Data Engineering,Unknown,Unknown,Unknown,Unknown,Unknown
154024923627,TEXT,ADVERTISER,Top Data Science Training,Unknown,Unknown,Unknown,Unknown,Unknown
154024923630,TEXT,ADVERTISER,Get Azure & AWS Skills Now,Unknown,Unknown,Unknown,Unknown,Unknown
154024923633,TEXT,ADVERTISER,Snowflake Expertise Awaits,Unknown,Unknown,Unknown,Unknown,Unknown
154024923636,TEXT,ADVERTISER,Career Boost in Data Science,Unknown,Unknown,Unknown,Unknown,Unknown
154024923639,TEXT,ADVERTISER,"Top data training with Azure, AWS, Snowflake expertise.",Unknown,Unknown,Unknown,Unknown,Unknown
154024923642,TEXT,ADVERTISER,Expert-led courses and career placement in data science.,Unknown,Unknown,Unknown,Unknown,Unknown
154024923645,TEXT,ADVERTISER,Gain skills in data engineering and secure top industry jobs.,Unknown,Unknown,Unknown,Unknown,Unknown
154024923648,TEXT,ADVERTISER,Unlock career potential with our data engineering training.,Unknown,Unknown,Unknown,Unknown,Unknown
154024980713,TEXT,ADVERTISER,Master Azure & AWS Skills,Unknown,Unknown,Unknown,Unknown,Unknown


In [0]:
"""
Displays the first 3 rows and schema of the conversion action DataFrame for inspection.
"""
display(conversion_action_df.limit(3))
conversion_action_df.printSchema()

conversion_action.id,conversion_action.name,conversion_action.status,conversion_action.category,conversion_action.primary_for_goal,conversion_action.counting_type,conversion_action.attribution_model_settings.attribution_model,conversion_action.click_through_lookback_window_days,conversion_action.view_through_lookback_window_days,conversion_action.include_in_conversions_metric
6858948899,academyofdata.in (web) purchase,HIDDEN,PURCHASE,false,MANY_PER_CLICK,GOOGLE_SEARCH_ATTRIBUTION_DATA_DRIVEN,90,1,false
6859177800,Calls from ads,ENABLED,PHONE_CALL_LEAD,true,MANY_PER_CLICK,GOOGLE_SEARCH_ATTRIBUTION_DATA_DRIVEN,30,30,true
6859177803,Clicks to call,ENABLED,CONTACT,true,MANY_PER_CLICK,GOOGLE_ADS_LAST_CLICK,30,7,false


root
 |-- conversion_action.id: string (nullable = true)
 |-- conversion_action.name: string (nullable = true)
 |-- conversion_action.status: string (nullable = true)
 |-- conversion_action.category: string (nullable = true)
 |-- conversion_action.primary_for_goal: string (nullable = true)
 |-- conversion_action.counting_type: string (nullable = true)
 |-- conversion_action.attribution_model_settings.attribution_model: string (nullable = true)
 |-- conversion_action.click_through_lookback_window_days: integer (nullable = true)
 |-- conversion_action.view_through_lookback_window_days: integer (nullable = true)
 |-- conversion_action.include_in_conversions_metric: string (nullable = true)



In [0]:
"""
Conversion action configuration data
"""
new_column_names = [c.replace('.', '_') for c in conversion_action_df.columns]

numeric_cols_to_fill = [
    'conversion_action_click_through_lookback_window_days',
    'conversion_action_view_through_lookback_window_days'
]
string_cols_to_fill = [
    'conversion_action_name', 'conversion_action_status', 'conversion_action_category',
    'conversion_action_primary_for_goal', 'conversion_action_counting_type',
    'conversion_action_attribution_model_settings_attribution_model',
    'conversion_action_include_in_conversions_metric'
]

df_conversion_action = conversion_action_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .select(
        "conversion_action_id",
        col("conversion_action_name").alias("name"),
        col("conversion_action_status").alias("status"),
        col("conversion_action_category").alias("category"),
        col("conversion_action_primary_for_goal").alias("primary_for_goal"),
        col("conversion_action_counting_type").alias("counting_type"),
        col("conversion_action_attribution_model_settings_attribution_model").alias("attribution_model"),
        col("conversion_action_click_through_lookback_window_days").alias("click_through_lookback_window_days"),
        col("conversion_action_view_through_lookback_window_days").alias("view_through_lookback_window_days"),
        col("conversion_action_include_in_conversions_metric").alias("include_in_conversions")
    )

print("--- Conversion Action Schema ---")
df_conversion_action.printSchema()
display(df_conversion_action.limit(10))

--- Conversion Action Schema ---
root
 |-- conversion_action_id: string (nullable = true)
 |-- name: string (nullable = false)
 |-- status: string (nullable = false)
 |-- category: string (nullable = false)
 |-- primary_for_goal: string (nullable = false)
 |-- counting_type: string (nullable = false)
 |-- attribution_model: string (nullable = false)
 |-- click_through_lookback_window_days: integer (nullable = false)
 |-- view_through_lookback_window_days: integer (nullable = false)
 |-- include_in_conversions: string (nullable = false)



conversion_action_id,name,status,category,primary_for_goal,counting_type,attribution_model,click_through_lookback_window_days,view_through_lookback_window_days,include_in_conversions
6858948899,academyofdata.in (web) purchase,HIDDEN,PURCHASE,false,MANY_PER_CLICK,GOOGLE_SEARCH_ATTRIBUTION_DATA_DRIVEN,90,1,false
6859177800,Calls from ads,ENABLED,PHONE_CALL_LEAD,true,MANY_PER_CLICK,GOOGLE_SEARCH_ATTRIBUTION_DATA_DRIVEN,30,30,true
6859177803,Clicks to call,ENABLED,CONTACT,true,MANY_PER_CLICK,GOOGLE_ADS_LAST_CLICK,30,7,false
6859183569,Book appointment,ENABLED,BOOK_APPOINTMENT,true,MANY_PER_CLICK,GOOGLE_SEARCH_ATTRIBUTION_DATA_DRIVEN,90,1,true
6864326819,Android installs (all other apps),ENABLED,DOWNLOAD,false,ONE_PER_CLICK,GOOGLE_ADS_LAST_CLICK,30,1,false
7052820665,Contact (Form submission https://academyofdata.ai/contact-us/contact-us/),ENABLED,CONTACT,true,MANY_PER_CLICK,GOOGLE_SEARCH_ATTRIBUTION_DATA_DRIVEN,30,1,true
7114762484,Contact,ENABLED,CONTACT,true,ONE_PER_CLICK,GOOGLE_ADS_LAST_CLICK,30,1,true
7117812172,Local actions - Directions,ENABLED,GET_DIRECTIONS,true,MANY_PER_CLICK,GOOGLE_ADS_LAST_CLICK,30,7,false
7122187177,Local actions - Other engagements,ENABLED,ENGAGEMENT,true,MANY_PER_CLICK,GOOGLE_ADS_LAST_CLICK,30,7,false
7123172631,Local actions - Website visits,ENABLED,PAGE_VIEW,true,MANY_PER_CLICK,GOOGLE_ADS_LAST_CLICK,30,7,false


In [0]:
"""
Displays the first 3 rows and schema of the placement performance DataFrame for inspection.
"""
display(placement_performance_df.limit(3))
placement_performance_df.printSchema()

campaign.id,campaign.name,campaign.advertising_channel_type,ad_group.id,ad_group.name,segments.date,detail_placement_view.placement,detail_placement_view.placement_type,detail_placement_view.display_name,detail_placement_view.group_placement_target_url,metrics.impressions,metrics.clicks,metrics.cost_micros,metrics.conversions,metrics.ctr,metrics.average_cpc
21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2024-10-15,2-com.differencetenderwhite.skirt,MOBILE_APPLICATION,"Mobile App: Block Puzzle Jewel (Google Play), by hua weiwei",https://play.google.com/store/apps/details?id=com.differencetenderwhite.skirt,5,0,0.0,0.0,0.0,null
21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2024-10-15,2-com.ht2.stick.hero,MOBILE_APPLICATION,"Mobile App: Stick Hero: Tower Defense (Google Play), by Unicorn Studio Official",https://play.google.com/store/apps/details?id=com.ht2.stick.hero,5,0,0.0,0.0,0.0,null
21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2024-10-15,2-com.ig.color.car.coloring,MOBILE_APPLICATION,"Mobile App: Car Coloring Pages ASMR (Google Play), by iKame Games - Zego Studio",https://play.google.com/store/apps/details?id=com.ig.color.car.coloring,16,0,0.0,0.0,0.0,null


root
 |-- campaign.id: string (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- campaign.advertising_channel_type: string (nullable = true)
 |-- ad_group.id: string (nullable = true)
 |-- ad_group.name: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- detail_placement_view.placement: string (nullable = true)
 |-- detail_placement_view.placement_type: string (nullable = true)
 |-- detail_placement_view.display_name: string (nullable = true)
 |-- detail_placement_view.group_placement_target_url: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.cost_micros: double (nullable = true)
 |-- metrics.conversions: double (nullable = true)
 |-- metrics.ctr: double (nullable = true)
 |-- metrics.average_cpc: double (nullable = true)



In [0]:
"""
Detailed placement performance (specific URLs/apps/videos)
"""
new_column_names = [c.replace('.', '_') for c in placement_performance_df.columns]

numeric_cols_to_fill = [
    'metrics_impressions', 'metrics_clicks', 'metrics_cost_micros',
    'metrics_conversions', 'metrics_ctr', 'metrics_average_cpc'
]
string_cols_to_fill = [
    'campaign_name', 'campaign_advertising_channel_type', 'ad_group_name',
    'detail_placement_view_placement', 'detail_placement_view_placement_type',
    'detail_placement_view_display_name', 'detail_placement_view_group_placement_target_url'
]

df_placement_performance = placement_performance_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("ctr_pct", col("metrics_ctr") * 100) \
    .withColumn("cost_inr", col("metrics_cost_micros") / 1000000) \
    .withColumn("cpc_inr", col("metrics_average_cpc") / 1000000) \
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        "campaign_advertising_channel_type",
        "ad_group_id",
        "ad_group_name",
        col("detail_placement_view_placement").alias("placement"),
        col("detail_placement_view_placement_type").alias("placement_type"),
        col("detail_placement_view_display_name").alias("display_name"),
        col("detail_placement_view_group_placement_target_url").alias("group_placement_target_url"),
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        "ctr_pct",
        "cpc_inr",
        "cost_inr"
    )

print("--- Placement Performance Schema ---")
df_placement_performance.printSchema()
display(df_placement_performance.limit(10))

--- Placement Performance Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- campaign_advertising_channel_type: string (nullable = false)
 |-- ad_group_id: string (nullable = true)
 |-- ad_group_name: string (nullable = false)
 |-- placement: string (nullable = false)
 |-- placement_type: string (nullable = false)
 |-- display_name: string (nullable = false)
 |-- group_placement_target_url: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- metrics_conversions: double (nullable = false)
 |-- ctr_pct: double (nullable = false)
 |-- cpc_inr: double (nullable = true)
 |-- cost_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,campaign_advertising_channel_type,ad_group_id,ad_group_name,placement,placement_type,display_name,group_placement_target_url,metrics_impressions,metrics_clicks,metrics_conversions,ctr_pct,cpc_inr,cost_inr
2024-10-15,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.differencetenderwhite.skirt,MOBILE_APPLICATION,"Mobile App: Block Puzzle Jewel (Google Play), by hua weiwei",https://play.google.com/store/apps/details?id=com.differencetenderwhite.skirt,5,0,0.0,0.0,0.0,0.0
2024-10-15,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.ht2.stick.hero,MOBILE_APPLICATION,"Mobile App: Stick Hero: Tower Defense (Google Play), by Unicorn Studio Official",https://play.google.com/store/apps/details?id=com.ht2.stick.hero,5,0,0.0,0.0,0.0,0.0
2024-10-15,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.ig.color.car.coloring,MOBILE_APPLICATION,"Mobile App: Car Coloring Pages ASMR (Google Play), by iKame Games - Zego Studio",https://play.google.com/store/apps/details?id=com.ig.color.car.coloring,16,0,0.0,0.0,0.0,0.0
2024-10-15,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.timebomb.gun.prank.simulator,MOBILE_APPLICATION,"Mobile App: Timebomb Prank, Gun Simulator (Google Play), by Era Global Publishing",https://play.google.com/store/apps/details?id=com.timebomb.gun.prank.simulator,6,0,0.0,0.0,0.0,0.0
2024-10-15,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.all.document.reader.doc.pdf.reader,MOBILE_APPLICATION,"Mobile App: All Document Reader: PDF, Word (Google Play), by Photo Video Team",https://play.google.com/store/apps/details?id=com.all.document.reader.doc.pdf.reader,5,0,0.0,0.0,0.0,0.0
2024-10-15,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.fundrivegames.mega.ramp.ultimate.cars.stunts.impossibletracks,MOBILE_APPLICATION,"Mobile App: Ramp Car Games: GT Car Stunts (Google Play), by Fun Drive Games",https://play.google.com/store/apps/details?id=com.fundrivegames.mega.ramp.ultimate.cars.stunts.impossibletracks,11,0,0.0,0.0,0.0,0.0
2024-10-15,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.pdfmerge.imagetopdf.pdfmaker,MOBILE_APPLICATION,"Mobile App: PDF Maker: Image to PDF (Google Play), by EZTech Apps",https://play.google.com/store/apps/details?id=com.pdfmerge.imagetopdf.pdfmaker,5,0,0.0,0.0,0.0,0.0
2024-10-15,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,1-898129576,MOBILE_APPLICATION,"Mobile App: Xender: Transfer, Share Files (iTunes App Store), by Xender (HK) Limited",https://itunes.apple.com/us/app/id898129576,6,0,0.0,0.0,0.0,0.0
2024-10-15,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,games.glance.com/play/Shape_Shifting-d81e05dd-007d-470b-9385-84b6d27c22c7,WEBSITE,games.glance.com/play/Shape_Shifting-d81e05dd-007d-470b-9385-84b6d27c22c7,glance.com,24,0,0.0,0.0,0.0,0.0
2024-10-15,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.gkgrips.babygames.toddler.games,MOBILE_APPLICATION,"Mobile App: Toddler Games: 2-5 Year Kids (Google Play), by Gkgrips",https://play.google.com/store/apps/details?id=com.gkgrips.babygames.toddler.games,5,0,0.0,0.0,0.0,0.0


In [0]:
"""
Displays the first 3 rows and schema of the group placement performance DataFrame for inspection.
"""
display(group_placement_performance_df.limit(3))
group_placement_performance_df.printSchema()

campaign.id,campaign.name,campaign.advertising_channel_type,ad_group.id,ad_group.name,segments.date,group_placement_view.placement,group_placement_view.placement_type,group_placement_view.display_name,metrics.impressions,metrics.clicks,metrics.cost_micros,metrics.conversions,metrics.ctr,metrics.average_cpc
21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2024-10-18,mobileapp::10002-ca-app-pub-3262303368935522,MOBILE_APPLICATION,mobileapp::10002-ca-app-pub-3262303368935522,2,1,145063.0,0.0,0.5,145063.0
21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2025-02-10,2-com.tangiappsit.shiva.racing.hero,MOBILE_APPLICATION,"Mobile App: Shiva Winter Biking Tales (Google Play), by TANGIAPPS IT SOLUTION PVT. LTD.",13,1,145009.0,0.0,0.07692307692307693,145009.0
21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2024-11-06,2-com.hmg.princess,MOBILE_APPLICATION,"Mobile App: Ice Princess Wedding Dress Up (Google Play), by Sweet Maker Shop",3,1,145004.0,0.0,0.3333333333333333,145004.0


root
 |-- campaign.id: string (nullable = true)
 |-- campaign.name: string (nullable = true)
 |-- campaign.advertising_channel_type: string (nullable = true)
 |-- ad_group.id: string (nullable = true)
 |-- ad_group.name: string (nullable = true)
 |-- segments.date: date (nullable = true)
 |-- group_placement_view.placement: string (nullable = true)
 |-- group_placement_view.placement_type: string (nullable = true)
 |-- group_placement_view.display_name: string (nullable = true)
 |-- metrics.impressions: integer (nullable = true)
 |-- metrics.clicks: integer (nullable = true)
 |-- metrics.cost_micros: double (nullable = true)
 |-- metrics.conversions: double (nullable = true)
 |-- metrics.ctr: double (nullable = true)
 |-- metrics.average_cpc: double (nullable = true)



In [0]:
"""
Group placement performance (aggregated placement groups)
"""
new_column_names = [c.replace('.', '_') for c in group_placement_performance_df.columns]

numeric_cols_to_fill = [
    'metrics_impressions', 'metrics_clicks', 'metrics_cost_micros',
    'metrics_conversions', 'metrics_ctr', 'metrics_average_cpc'
]
string_cols_to_fill = [
    'campaign_name', 'campaign_advertising_channel_type', 'ad_group_name',
    'group_placement_view_placement', 'group_placement_view_placement_type',
    'group_placement_view_display_name'
]

df_group_placement_performance = group_placement_performance_df.toDF(*new_column_names) \
    .fillna(0, subset=numeric_cols_to_fill) \
    .fillna('Unknown', subset=string_cols_to_fill) \
    .withColumn("ctr_pct", col("metrics_ctr") * 100) \
    .withColumn("cost_inr", col("metrics_cost_micros") / 1000000) \
    .withColumn("cpc_inr", col("metrics_average_cpc") / 1000000) \
    .select(
        "segments_date",
        "campaign_id",
        "campaign_name",
        "campaign_advertising_channel_type",
        "ad_group_id",
        "ad_group_name",
        col("group_placement_view_placement").alias("placement"),
        col("group_placement_view_placement_type").alias("placement_type"),
        col("group_placement_view_display_name").alias("display_name"),
        "metrics_impressions",
        "metrics_clicks",
        "metrics_conversions",
        "ctr_pct",
        "cpc_inr",
        "cost_inr"
    )

print("--- Group Placement Performance Schema ---")
df_group_placement_performance.printSchema()
display(df_group_placement_performance.limit(10))

--- Group Placement Performance Schema ---
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = false)
 |-- campaign_advertising_channel_type: string (nullable = false)
 |-- ad_group_id: string (nullable = true)
 |-- ad_group_name: string (nullable = false)
 |-- placement: string (nullable = false)
 |-- placement_type: string (nullable = false)
 |-- display_name: string (nullable = false)
 |-- metrics_impressions: integer (nullable = false)
 |-- metrics_clicks: integer (nullable = false)
 |-- metrics_conversions: double (nullable = false)
 |-- ctr_pct: double (nullable = false)
 |-- cpc_inr: double (nullable = true)
 |-- cost_inr: double (nullable = true)



segments_date,campaign_id,campaign_name,campaign_advertising_channel_type,ad_group_id,ad_group_name,placement,placement_type,display_name,metrics_impressions,metrics_clicks,metrics_conversions,ctr_pct,cpc_inr,cost_inr
2024-10-18,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,mobileapp::10002-ca-app-pub-3262303368935522,MOBILE_APPLICATION,mobileapp::10002-ca-app-pub-3262303368935522,2,1,0.0,50.0,0.145063,0.145063
2025-02-10,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.tangiappsit.shiva.racing.hero,MOBILE_APPLICATION,"Mobile App: Shiva Winter Biking Tales (Google Play), by TANGIAPPS IT SOLUTION PVT. LTD.",13,1,0.0,7.6923076923076925,0.145009,0.145009
2024-11-06,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.hmg.princess,MOBILE_APPLICATION,"Mobile App: Ice Princess Wedding Dress Up (Google Play), by Sweet Maker Shop",3,1,0.0,33.33333333333333,0.145004,0.145004
2024-11-06,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-air.com.winkypinky.lunchboxcookinganddecoration,MOBILE_APPLICATION,"Mobile App: Lunch Box Cooking & Decoration (Google Play), by winkypinky",2,1,0.0,50.0,0.145004,0.145004
2024-11-06,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-sweet.selfie.beauty.camera.ar,MOBILE_APPLICATION,"Mobile App: Beauty Camera & Makeup Stylist (Google Play), by Coocent",2,1,0.0,50.0,0.145004,0.145004
2024-11-06,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.tangiappsit.shiva.racing.hero,MOBILE_APPLICATION,"Mobile App: Shiva Winter Biking Tales (Google Play), by TANGIAPPS IT SOLUTION PVT. LTD.",1,1,0.0,100.0,0.145004,0.145004
2025-02-11,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.fullmetalgamedev.coffeerun3d,MOBILE_APPLICATION,"Mobile App: Coffee Dash 3D (Google Play), by fullmetalgamedev",21,1,0.0,4.761904761904762,0.144901,0.144901
2024-10-22,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.halo.wifikey.wifilocating,MOBILE_APPLICATION,"Mobile App: WiFi Master: WiFi Auto Connect (Google Play), by LINKSURE NETWORK HOLDING",1,1,0.0,100.0,0.144894,0.144894
2024-10-22,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-ins.story.unfold,MOBILE_APPLICATION,"Mobile App: Mivo: Face Swap Video Bride (Google Play), by Mivo studio",2,1,0.0,50.0,0.14489,0.14489
2024-10-18,21771245709,Azure Data Engineering -Display - Image,DISPLAY,169993022084,Ad group 1,2-com.allindiaradio.fmradio.newsonair,MOBILE_APPLICATION,"Mobile App: FM Radio - Radio India HD (Google Play), by Radio Lite Apps",2,1,0.0,50.0,0.144889,0.144889


In [0]:
"""
Ensures the silver database exists and saves all transformed DataFrames to the silver layer as Delta tables.
"""
# Ensure database exists
spark.sql("CREATE DATABASE IF NOT EXISTS googleads_silver")

dataframes_to_save = {
    "campaigns": df_core_campaign,
    "keywords": df_search_keyword,
    "search_terms": df_search_term,
    "conversions": df_conversion,
    "ad_copy": df_ad_copy,
    "devices": df_optimization_device,
    "genders": df_audience_gender,
    "ages": df_audience_age,
    "impressions": df_competitive_impression,
    "networks": df_network,
    "display_ads": df_display_ad,
    "calls": df_call_lead_generation,
    "locations": df_geographic_enriched,
    "geo_constants": df_geo_target_constant,
    "ads_data":df_ads_data,
    "ads_performance":df_ads_performance,
    "campaign_assets": df_campaign_asset,
    "ad_group_ad_assets": df_ad_group_ad_asset_view,
    "assets": df_asset,
    "conversion_actions": df_conversion_action,
    "placements": df_placement_performance,
    "group_placements": df_group_placement_performance  
}

# Loop through the dictionary and save each DataFrame in googleads_silver
for table_name, df in dataframes_to_save.items():
    try:
        full_table_name = f"googleads_silver.{table_name}"
        print(f"--- Saving {full_table_name}... ---")

        df.write.format("delta") \
            .option("overwriteSchema", "true")\
            .mode("overwrite") \
            .saveAsTable(full_table_name)

        print(f"SUCCESS: DataFrame successfully saved as Delta table '{full_table_name}'.")

    except Exception as e:
        print(f"ERROR saving {full_table_name}: {e}")

print("\n--- All DataFrames have been processed. ---")


--- Saving googleads_silver.campaigns... ---
SUCCESS: DataFrame successfully saved as Delta table 'googleads_silver.campaigns'.
--- Saving googleads_silver.keywords... ---
SUCCESS: DataFrame successfully saved as Delta table 'googleads_silver.keywords'.
--- Saving googleads_silver.search_terms... ---
SUCCESS: DataFrame successfully saved as Delta table 'googleads_silver.search_terms'.
--- Saving googleads_silver.conversions... ---
SUCCESS: DataFrame successfully saved as Delta table 'googleads_silver.conversions'.
--- Saving googleads_silver.ad_copy... ---
SUCCESS: DataFrame successfully saved as Delta table 'googleads_silver.ad_copy'.
--- Saving googleads_silver.devices... ---
SUCCESS: DataFrame successfully saved as Delta table 'googleads_silver.devices'.
--- Saving googleads_silver.genders... ---
SUCCESS: DataFrame successfully saved as Delta table 'googleads_silver.genders'.
--- Saving googleads_silver.ages... ---
SUCCESS: DataFrame successfully saved as Delta table 'googleads_silv

In [0]:
%sql SHOW COLUMNS IN googleads_silver.ads_data

col_name
campaign_id
campaign_name
ad_group_id
ad_group_name
ad_group_ad_ad_id
ad_group_ad_ad_type
ad_group_ad_status
ad_group_ad_ad_final_urls
ad_group_ad_ad_responsive_search_ad_headlines
ad_group_ad_ad_responsive_search_ad_descriptions


In [0]:
# 1. Configuration
catalog_name = "workspace" 
schema_name = "googleads_silver"
full_schema_path = f"{catalog_name}.{schema_name}"

print(f"{'='*60}")
print(f"Source: {full_schema_path}")
print(f"{'='*60}\n")

try:
    # Fetch list of tables from Unity Catalog
    tables = spark.catalog.listTables(full_schema_path)
    
    total_rows_all_tables = 0
    table_count = 0

    for table in tables:
        full_table_name = f"{full_schema_path}.{table.name}"
        
        try:
            # Load the table strictly from the Metastore/Storage
            df_audit = spark.table(full_table_name)
            
            # 1. Get Row Count
            # [VALIDATED] .count() forces a read of the Delta log/parquet files
            row_count = df_audit.count()
            total_rows_all_tables += row_count
            table_count += 1
            
            # 2. Print Report Block
            print(f"\n{'-'*20} TABLE: {table.name} {'-'*20}")
            print(f"LOCATION: {full_table_name}")
            print(f"ROW COUNT: {row_count:,}")
            
            print(f"{'-'*10} SCHEMA {'-'*10}")
            df_audit.printSchema()
            print(f"{'='*60}\n")
            
        except Exception as e:
            print(f"[ERROR] Could not read table {table.name}: {e}\n")

    # Final Summary
    print(f"SUMMARY:")
    print(f"Total Tables Inspected: {table_count}")
    print(f"Total Rows in Silver Layer: {total_rows_all_tables:,}")

except Exception as e:
    print(f"[CRITICAL ERROR] Could not list tables in {full_schema_path}.")
    print(f"Error Details: {e}")

Source: workspace.googleads_silver


-------------------- TABLE: ad_copy --------------------
LOCATION: workspace.googleads_silver.ad_copy
ROW COUNT: 275
---------- SCHEMA ----------
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- ad_group_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- ad_group_name: string (nullable = true)
 |-- ad_id: string (nullable = true)
 |-- metrics_impressions: integer (nullable = true)
 |-- metrics_clicks: integer (nullable = true)
 |-- metrics_conversions: double (nullable = true)
 |-- ctr_pct: double (nullable = true)
 |-- cvr_pct: double (nullable = true)
 |-- final_url: string (nullable = true)
 |-- domain: string (nullable = true)



-------------------- TABLE: ad_group_ad_assets --------------------
LOCATION: workspace.googleads_silver.ad_group_ad_assets
ROW COUNT: 5,595
---------- SCHEMA ----------
root
 |-- segments_date: date (nullable = true)
 |-- campaign_id: string (nu

# Google Ads Metric Definitions

## 1. Fundamental Metrics

### **Impressions**
* **Definition:** The number of times your advertisement was displayed on a screen. This is the base metric for visibility.
* **Context:** An impression is counted each time your ad is shown on a search result page or other site on the Google Network.

### **CPC (Cost Per Click)**
* **Definition:** The actual price you pay for each click in your pay-per-click (PPC) marketing campaigns.
* **Formula:** $$CPC = \frac{\text{Total Cost}}{\text{Number of Clicks}}$$
* **Example:** If you spend $100 and get 50 clicks, your CPC is $2.00.

### **CTR (Click-Through Rate)**
* **Definition:** The ratio of users who click on a specific link to the number of total users who view a page, email, or advertisement. It measures the relevance of your ad.
* **Formula:** $$CTR = \left( \frac{\text{Number of Clicks}}{\text{Number of Impressions}} \right) \times 100$$
* **Example:** If your ad had 1,000 impressions and one click, the CTR is 0.1%.

---

## 2. Conversion & Efficiency Metrics

### **CVR (Conversion Rate)**
* **Definition:** The average number of conversions per ad interaction, shown as a percentage. It measures how effective your landing page is at turning visitors into customers.
* **Formula:** $$CVR = \left( \frac{\text{Number of Conversions}}{\text{Number of Clicks}} \right) \times 100$$
* **Example:** If you had 50 clicks and 2 sales (conversions), your CVR is 4%.

### **CPA (Cost Per Acquisition / Action)**
* **Definition:** The average amount you have been charged for a conversion from your ad.
* **Formula:** $$CPA = \frac{\text{Total Cost}}{\text{Number of Conversions}}$$
* **Example:** If you spent $100 and got 2 sales, your CPA is $50.

### **ROAS (Return on Ad Spend)**
* **Definition:** A marketing metric that measures the amount of revenue your business earns for each dollar it spends on advertising.
* **Formula:** $$ROAS = \frac{\text{Revenue from Ads}}{\text{Cost of Ads}}$$
* **Example:** If you spent $100 on ads and made $500 in sales, your ROAS is 5 (or 500%).